# Ceiling Test Suite — Paper 1 & Paper 2
**Run every cell top to bottom** (Runtime → Run all). It downloads real Binance data, checks the theorems numerically, runs every test in the revision plan, and writes each result to `results/<TEST_ID>.json` with one line: **PASS**, **FAIL** or **REPORT**.

**Rules this notebook enforces**
1. **No typed-in results.** Every number in either paper must come from a `results/*.json` file made here. Cell A1 lists the numbers your old notebook typed in by hand (cell 84's crash CIs, the `40.79 * ...` curve, the `M_C: 22.87 / 14.16` dicts). Do not reuse any of them.
2. **Labels travel with numbers.** Each result is tagged `theorem-check`, `simulation`, `real-data` or `audit`. The paper must use the same word.
3. **FAIL is information, not a reason to tune.** If a real-data test fails, report it and change the claim (see the gate rules in the plan doc).
4. **The 2024 test year is opened once.** Cell P2.5a writes the pre-registration; P2.5b counts every time the test set is evaluated.

**Run time:** about 30–60 min the first time (downloads ~110 days of 1-second data); a few minutes after that (cached). Set `USE_DRIVE = True` in cell 0 to keep data across Colab resets.

**Upload first:** your newest Paper 1 and Paper 2 PDFs (named and anonymous) to `/content`.


## 0. Setup

In [ ]:
# ============================================================================
# CELL 0 — SETUP (run first, every session)
# Creates folders, installs packages, defines record() which writes every
# result to results/<TEST_ID>.json and prints one PASS / FAIL / REPORT line.
#
# Result kinds (they are printed and saved, and the paper must use the same label):
#   "theorem-check"  numerical check of a theorem on simulated systems
#   "simulation"     property of an estimator / controller in a simulated market
#   "real-data"      computed from downloaded Binance files in THIS session
#   "audit"          checks on the manuscript, code or repository
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, itertools
import urllib.request

IN_COLAB = "google.colab" in sys.modules

def _pip(pkg, mod=None):
    try:
        __import__(mod or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for _p, _m in [("pymupdf", "fitz"), ("pyarrow", "pyarrow"), ("scipy", "scipy"), ("pandas", "pandas")]:
    _pip(_p, _m)

import numpy as np
import pandas as pd
from scipy import linalg, optimize, stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- Folders --------------------------------------------------------------
# Set USE_DRIVE = True to keep downloaded data and results across Colab resets.
USE_DRIVE = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")

RESULTS = os.path.join(BASE, "results")
DATA = os.path.join(BASE, "data")
FIGS = os.path.join(BASE, "figures")
PREREG = os.path.join(BASE, "prereg")
for _d in (RESULTS, DATA, FIGS, PREREG):
    os.makedirs(_d, exist_ok=True)

SEED = 20261018

def rng(offset=0):
    """Independent, reproducible random generator per test."""
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, float):
        return None if not math.isfinite(o) else o
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    """passed: True / False / None (None = REPORT: a result reported whatever it is)."""
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {
        "test_id": test_id, "status": status, "kind": kind,
        "criterion": criterion, "note": note, "metrics": _clean(metrics),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "seed": SEED,
    }
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def loglog_slope(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = (x > 0) & (y > 0) & np.isfinite(y)
    b = np.polyfit(np.log(x[ok]), np.log(y[ok]), 1)
    return float(b[0])

print("IN_COLAB =", IN_COLAB, "| results ->", RESULTS)


In [ ]:
# ============================================================================
# CELL 1 — REPOSITORY AND MANUSCRIPT FILES
# Clones your repo (for the audit cells) and points to the current PDFs.
# Upload the newest PDFs to /content (left sidebar → Files → upload) or Drive,
# then set the four paths below.
# ============================================================================
REPO = "/content/Fianince" if IN_COLAB else os.path.abspath("repo")
if IN_COLAB and not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Pratikn03/Fianince.git", REPO], check=True)

def _first(*cands):
    for c in cands:
        hits = sorted(glob.glob(c))
        if hits:
            return hits[-1]
    return None

_UP = "/content" if IN_COLAB else "/mnt/user-data/uploads"
PAPER1_PDF = _first(f"{_UP}/Paper_1*CLEAN*.pdf", f"{_UP}/Paper_1*.pdf")
PAPER2_PDF = _first(f"{_UP}/Paper_2*Named*CLEAN*.pdf", f"{_UP}/Paper_2*Named*.pdf", f"{_UP}/Paper_2*.pdf")
PAPER2_BLIND_PDF = _first(f"{_UP}/Paper_2*Anonymous*.pdf")
# Your LaTeX sources (used by the number-consistency audit). Adjust if needed.
PAPER1_TEX = _first(f"{REPO}/paper1/*.tex", f"{REPO}/paper/*Paper_1*.tex")
PAPER2_TEX = _first(f"{REPO}/paper2/*.tex", f"{REPO}/paper/*Paper_2*.tex")

for k in ["REPO", "PAPER1_PDF", "PAPER2_PDF", "PAPER2_BLIND_PDF", "PAPER1_TEX", "PAPER2_TEX"]:
    print(f"{k:18s} = {globals()[k]}")

def pdf_pages_text(path):
    import fitz
    with fitz.open(path) as doc:
        return [p.get_text() for p in doc]


In [ ]:
# ============================================================================
# CELL 2 — REAL DATA LOADER (Binance public 1-second klines, checksum-verified)
# Every real-data test downloads through load_day(). There is NO synthetic
# fallback: if a download fails, the test stops and says so.
# Data licence: Binance public data (data.binance.vision); cite it in the paper.
# ============================================================================
BINANCE = "https://data.binance.vision/data/spot/daily/klines/{sym}/1s/{sym}-1s-{date}.zip"
KLINE_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
              "quote_volume", "n_trades", "taker_buy_base", "taker_buy_quote", "ignore"]

DOWNLOAD_LOG = []

def _get(url, timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except Exception as e:
            last = e
            time.sleep(2 * (i + 1))
    raise RuntimeError(f"download failed: {url}: {last}")

def load_day(sym, date):
    """One UTC day of 1s klines -> DataFrame indexed by second (86,400 rows):
       close (USDT), flow_q (signed taker quote flow, USDT), flow_b (signed base flow),
       vol_q (quote volume). Missing seconds: close forward-filled, flows 0."""
    pq = os.path.join(DATA, f"{sym}_{date}.parquet")
    if os.path.exists(pq):
        return pd.read_parquet(pq)
    url = BINANCE.format(sym=sym, date=date)
    raw = _get(url)
    sha = hashlib.sha256(raw).hexdigest()
    official = _get(url + ".CHECKSUM").decode().split()[0]
    if sha != official:
        raise RuntimeError(f"SHA-256 mismatch for {sym} {date}: {sha} vs {official}")
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f, header=None, names=KLINE_COLS)
    t = df["open_time"].astype("int64")
    t = np.where(t > 1e14, t // 1_000_000, t // 1000)  # 2025+ files use microseconds
    s = pd.DataFrame({
        "close": df["close"].astype(float).to_numpy(),
        "flow_q": (2 * df["taker_buy_quote"] - df["quote_volume"]).astype(float).to_numpy(),
        "flow_b": (2 * df["taker_buy_base"] - df["volume"]).astype(float).to_numpy(),
        "vol_q": df["quote_volume"].astype(float).to_numpy(),
    }, index=pd.Index(t, name="sec"))
    s = s[~s.index.duplicated()]
    t0 = int(pd.Timestamp(f"{date} 00:00:00", tz="UTC").timestamp())
    s = s.reindex(np.arange(t0, t0 + 86400))
    s["close"] = s["close"].ffill().bfill()
    s[["flow_q", "flow_b", "vol_q"]] = s[["flow_q", "flow_b", "vol_q"]].fillna(0.0)
    s.index.name = "sec"
    s.to_parquet(pq)
    DOWNLOAD_LOG.append({"sym": sym, "date": date, "sha256": sha})
    return s

def load_days(sym, dates, verbose=True):
    out = {}
    for i, d in enumerate(dates):
        try:
            out[d] = load_day(sym, d)
        except Exception as e:
            print(f"   ! {sym} {d}: {e}")
        if verbose and (i + 1) % 10 == 0:
            print(f"   loaded {i+1}/{len(dates)} days")
    return out

def bars(day, delta=1):
    """Aggregate to delta-second bars. Returns log-price in bp, flow in $1M."""
    if delta == 1:
        p = np.log(day["close"].to_numpy()) * 1e4
        f = day["flow_q"].to_numpy() / 1e6
        v = day["vol_q"].to_numpy() / 1e6
        return p, f, v
    n = len(day) // delta * delta
    c = day["close"].to_numpy()[:n].reshape(-1, delta)[:, -1]
    f = day["flow_q"].to_numpy()[:n].reshape(-1, delta).sum(1) / 1e6
    v = day["vol_q"].to_numpy()[:n].reshape(-1, delta).sum(1) / 1e6
    return np.log(c) * 1e4, f, v

def ema_dev(p, span):
    """Price deviation from its EMA (the paper's x_t), bp."""
    return p - pd.Series(p).ewm(span=span, adjust=False).mean().to_numpy()

# ---- Event and panel calendars (fixed BEFORE looking at results) -----------
CRASH_DAYS = {                       # stress days used in Paper 2
    "May2021": "2021-05-19", "LUNA2022": "2022-05-09",
    "FTX2022": "2022-11-08", "SVB2023": "2023-03-11",
}
OOS_CRASH_DAYS = {"Aug2024": "2024-08-05"}   # held out: never used for tuning
PANEL_DAYS = [f"{y}-{m:02d}-{d:02d}" for y in (2021, 2022, 2023, 2024)
              for m in range(1, 13) for d in (10, 24)]
TRAIN_YEARS, TEST_YEARS = (2021, 2022, 2023), (2024,)
HOLDOUT_JAN2023 = [f"2023-01-{d:02d}" for d in range(10, 21)]
print(f"{len(PANEL_DAYS)} panel days; crash days: {list(CRASH_DAYS.values())}")


## A. Integrity and manuscript audits
A1 must be clean before any paper is posted. A2–A3 are Phase 0 of the plan (split, recompile, anonymity).

In [ ]:
# ============================================================================
# CELL A1 — HARD-CODED RESULT SCAN  (integrity check; run before anything else)
# Finds headline numbers from the papers typed into code as literals.
# Every hit must be either deleted or replaced by code that computes it.
# ============================================================================
KEY_NUMBERS = ["34.54", "16.67", "3.55", "2.37", "14.16", "22.87", "40.79",
               "25.31", "26.52", "0.906", "0.904", "9.83", "29.59", "0.2562",
               "98.3", "66.19", "385.05", "1.98", "19.42", "9.84", "43.80", "26.10"]
# Upload your OLD notebook (e.g. Untitled8.ipynb) to /content so it is scanned too.
NOTEBOOKS_TO_SCAN = [p for p in (glob.glob("/content/*.ipynb") if IN_COLAB else glob.glob("/mnt/user-data/uploads/*.ipynb"))
                     if "Ceiling_Test_Suite" not in p]

def _code_lines():
    for path in glob.glob(f"{REPO}/**/*.py", recursive=True):
        if "/.git/" in path or "/archive/" in path:
            continue
        try:
            for i, line in enumerate(open(path, errors="ignore"), 1):
                yield path, i, line
        except Exception:
            pass
    for nb in NOTEBOOKS_TO_SCAN:
        try:
            cells = json.load(open(nb))["cells"]
        except Exception:
            continue
        for ci, c in enumerate(cells):
            if c["cell_type"] != "code":
                continue
            for i, line in enumerate("".join(c["source"]).split("\n"), 1):
                yield f"{os.path.basename(nb)}[cell {ci}]", i, line

hits = []
pat = re.compile(r"(?<![\d.])(" + "|".join(re.escape(k) for k in KEY_NUMBERS) + r")(?![\d])")
for path, i, line in _code_lines():
    s = line.strip()
    if s.startswith("#"):
        continue
    m = pat.search(s)
    if m:
        # "typed_value": number assigned as data (e.g. "mean": 34.54 or x = 14.16)
        # "typed_into_latex": number written by hand into manuscript text by code
        n = re.escape(m.group(1))
        if re.search(r"[:=\[,(]\s*" + n + r"\s*[,}\])]?", s) and "\\" not in s and "$" not in s:
            cat = "typed_value"
        else:
            cat = "typed_into_latex"
        hits.append({"where": f"{path}:{i}", "number": m.group(1), "category": cat, "code": s[:160]})

by_file = pd.Series([h["where"].rsplit(":", 1)[0] for h in hits]).value_counts() if hits else pd.Series(dtype=int)
typed = [h for h in hits if h["category"] == "typed_value"]
latex = [h for h in hits if h["category"] == "typed_into_latex"]
print(f"{len(typed)} headline numbers typed as DATA values; {len(latex)} typed by hand into LaTeX text")
print("\n--- typed as data values (most serious: a result that is not computed) ---")
for h in typed[:60]:
    print(f"  {h['where']:60s} {h['number']:>7s} | {h['code'][:95]}")
record("A1_hardcoded_scan", len(typed) == 0, "audit",
       {"n_typed_values": len(typed), "n_typed_into_latex": len(latex),
        "typed_values": typed, "typed_into_latex": latex, "files": by_file.to_dict()},
       criterion="no headline result typed into code as a data value",
       note="Fix: every number in the papers must be read from results/*.json produced by a test cell.")


In [ ]:
# ============================================================================
# CELL A2 — TEXT OVERLAP BETWEEN PAPER 1 AND PAPER 2  (plan S1-T1)
# Pass: < 5% of Paper 2's long sentences appear verbatim in Paper 1.
# ============================================================================
def _sentences(pages):
    txt = " ".join(pages)
    txt = re.sub(r"-\s*\n\s*", "", txt)
    txt = re.sub(r"\s+", " ", txt)
    out = []
    for s in re.split(r"(?<=[.!?])\s+", txt):
        norm = re.sub(r"[^a-z0-9]", "", s.lower())
        if len(s) > 80 and len(norm) > 60:
            out.append(norm)
    return out

def _split_main_appendix(pages):
    for i, p in enumerate(pages):
        if re.search(r"Internet Appendix", p) and i > len(pages) * 0.3:
            return pages[:i], pages[i:]
    return pages, []

if PAPER1_PDF and PAPER2_PDF:
    p1, p2 = pdf_pages_text(PAPER1_PDF), pdf_pages_text(PAPER2_PDF)
    s1 = set(_sentences(p1))
    p2_main, p2_app = _split_main_appendix(p2)
    s2_main, s2_app = _sentences(p2_main), _sentences(p2_app)
    ov_main = np.mean([s in s1 for s in s2_main]) if s2_main else 0.0
    ov_app = np.mean([s in s1 for s in s2_app]) if s2_app else 0.0
    ov_all = np.mean([s in s1 for s in s2_main + s2_app])
    print(f"Paper 2 sentences found verbatim in Paper 1: main {ov_main:.1%}, appendix {ov_app:.1%}, all {ov_all:.1%}")
    record("S1-T1_overlap", bool(ov_all < 0.05 and ov_app == 0), "audit",
           {"overlap_main": ov_main, "overlap_appendix": ov_app, "overlap_all": ov_all,
            "n_p2_sentences": len(s2_main) + len(s2_app)},
           criterion="< 5% overall and 0% in appendices")
else:
    print("Upload both PDFs and re-run CELL 1.")


In [ ]:
# ============================================================================
# CELL A3 — PAGE HEADERS, PAGE NUMBERS, ANONYMITY, LEAKS  (S1-T2, S2-T1, S3-T1)
# ============================================================================
NAMES = ["Niroula", "Alkhushayni", "Bhavsar", "Chen", "mnsu", "Mankato", "Tribhuvan", "Pratikn03"]
P2_ONLY_TERMS = ["Kreiss", "resolvent guard", "liquidation rate", "G_Kreiss", "Resolvent Solvency"]

def _page_number_issues(pages):
    bad = []
    for i, t in enumerate(pages, 1):
        tail = "\n".join(t.strip().split("\n")[-6:])
        nums = re.findall(r"(?m)^\s*(?:IA\.)?(\d{1,3})\s*$", tail)
        if len(nums) != 1:
            bad.append({"page": i, "numbers_found": nums})
    return bad

def _double_header(pages):
    bad = []
    for i, t in enumerate(pages, 1):
        head = " ".join(t.strip().split("\n")[:4])
        if re.search(r"(\b\w+\b)\s+(?:and\s+)?\1\b.*\1", head) or head.count("Niroula") >= 3:
            bad.append(i)
    return bad

checks = {}
for label, path in [("paper1", PAPER1_PDF), ("paper2_named", PAPER2_PDF), ("paper2_blind", PAPER2_BLIND_PDF)]:
    if not path:
        continue
    pages = pdf_pages_text(path)
    full = "\n".join(pages)
    c = {"pages": len(pages),
         "page_number_issues": _page_number_issues(pages)[:15],
         "n_page_number_issues": len(_page_number_issues(pages)),
         "double_header_pages": _double_header(pages)[:15],
         "double_blind_banner": "Double-Blind" in full}
    if label == "paper2_blind":
        c["name_leaks"] = {n: full.count(n) for n in NAMES if n in full}
    if label == "paper1":
        c["paper2_terms_in_paper1"] = {t: full.count(t) for t in P2_ONLY_TERMS if t in full}
    checks[label] = c
    print(label, json.dumps({k: v for k, v in c.items() if k != "page_number_issues"}, default=str))

if checks:
    ok_pages = all(v["n_page_number_issues"] == 0 and not v["double_header_pages"] for v in checks.values())
    record("S2-T1_page_layout", ok_pages, "audit", checks,
           criterion="every page has exactly one page number and one header")
    if "paper2_blind" in checks:
        record("S3-T1_anonymity", not checks["paper2_blind"].get("name_leaks"), "audit",
               checks["paper2_blind"], criterion="zero author names / affiliations in blind PDF")
    if "paper2_named" in checks:
        record("S3-T1b_named_has_no_blind_banner", not checks["paper2_named"]["double_blind_banner"], "audit",
               {"double_blind_banner": checks["paper2_named"]["double_blind_banner"]},
               criterion="named version does not say 'Double-Blind Submission'")
    if "paper1" in checks:
        record("S1-T2_leak_check", not checks["paper1"].get("paper2_terms_in_paper1"), "audit",
               checks["paper1"], criterion="no Paper 2 guard/liquidation content in Paper 1")


## T. Theorem checks (simulation)
Numerical checks of the theorems in the plan doc. A PASS means the statement held in every simulated case, which tells you it is worth proving. It is **not** a proof. Write the proofs from the proof routes in the plan.

Note on **T2**: the check corrected the plan's first draft. Signed cohorts do *not* break minimum phase, and passive noise *helps* identification. The barrier is cost (T3), not impossibility.

In [ ]:
# ============================================================================
# CELL T0 — THE COHORT ECONOMY OF PAPER 1 (shared by all theorem checks)
#   x_t = a x_{t-1} + kappa * sum_i q_i y_{i,t-1} + kappa v_t
#   y_{i,t} = (1 - eta_i) y_{i,t-1} + eta_i x_t
# Merged measure: weights w_i = q_i * eta_i on memory rates r_i = 1 - eta_i,
# feedback kernel k_m = sum_i w_i r_i^m  (G(z) = sum_i w_i z / (z - r_i)).
# ============================================================================
def impulse_from_kernel(k, a, kappa, L):
    """Price impulse response h_0..h_{L-1} to v_0 = 1, given kernel k_m = sum w_i r_i^m.
    Uses g_t = sum_{s<=t} k_{t-s} h_s and h_{t+1} = a h_t + kappa g_t."""
    k = np.asarray(k, float)
    h = np.zeros(L)
    h[0] = kappa
    for t in range(L - 1):
        g = np.dot(k[:t + 1][::-1], h[:t + 1])
        h[t + 1] = a * h[t] + kappa * g
    return h

def kernel(w, r, L):
    m = np.arange(L)[:, None]
    return (np.asarray(w)[None, :] * np.asarray(r)[None, :] ** m).sum(1)

def impulse(w, r, a, kappa, L):
    return impulse_from_kernel(kernel(w, r, L), a, kappa, L)

def kernel_from_impulse(h, a, kappa):
    """Deconvolution: recover k from h (lower-triangular Toeplitz solve)."""
    L = len(h)
    g = (h[1:] - a * h[:-1]) / kappa
    T = linalg.toeplitz(h[:L - 1], np.zeros(L - 1))
    return linalg.solve_triangular(T, g, lower=True)

def is_stable(w, r, a, kappa):
    """Closed-loop eigenvalues of the (K+1)-state system."""
    K = len(w)
    eta = 1 - np.asarray(r)
    q = np.asarray(w) / eta
    A = np.zeros((K + 1, K + 1))
    A[0, 0] = a
    A[0, 1:] = kappa * q
    for i in range(K):           # y_i,t = r_i y_i,t-1 + eta_i x_t  (substitute x_t)
        A[1 + i, :] = eta[i] * A[0, :]
        A[1 + i, 1 + i] += r[i]
    return float(np.max(np.abs(np.linalg.eigvals(A)))) < 1

def matched_pair(K, seed=0):
    """Two K-atom positive measures on [-1,1] whose moments 0..2K-2 agree and
    whose moment 2K-1 differs (Gauss rule vs Gauss-Radau rule with a fixed node).
    Returns (sA, wA), (sB, wB) and the moment gap at order 2K-1."""
    sA, wA = np.polynomial.legendre.leggauss(K)
    wA = wA / wA.sum()
    target = np.array([np.sum(wA * sA ** j) for j in range(2 * K - 1)])
    fixed = -1.0                                # Radau: one node fixed at -1
    def resid(z):
        s = np.concatenate([[fixed], z[:K - 1]])
        w = z[K - 1:]
        return np.array([np.sum(w * s ** j) for j in range(2 * K - 1)]) - target
    z0 = np.concatenate([np.linspace(-0.5, 0.9, K - 1), np.full(K, 1.0 / K)])
    sol = optimize.least_squares(resid, z0, xtol=1e-15, ftol=1e-15, gtol=1e-15)
    sB = np.concatenate([[fixed], sol.x[:K - 1]])
    wB = sol.x[K - 1:]
    gap = np.sum(wB * sB ** (2 * K - 1)) - np.sum(wA * sA ** (2 * K - 1))
    assert np.max(np.abs(resid(sol.x))) < 1e-10, "moment matching failed"
    assert np.all(wB > 0) and len(np.unique(np.round(sB, 8))) == K, "Radau rule invalid"
    return (sA, wA), (sB, wB), float(gap)

def power_norms(J, K=3000):
    out, P = [], np.eye(len(J))
    for _ in range(K):
        out.append(np.linalg.norm(P, 2)); P = P @ J
        if out[-1] < 1e-14:
            break
    return np.array(out)

A_DEF, KAPPA_DEF, W_TOTAL, R_CENTER = 0.5, 1.0, 0.10, 0.70
print("cohort model helpers ready")


In [ ]:
# ============================================================================
# CELL T1 — GENERAL K-COHORT RATE (lower bound side)
# Claim to check: two K-cohort economies whose merged measures agree in moments
# 0..2K-2 have price responses differing by ||h_A - h_B|| ∝ (Δη)^(2K-1), so the
# Le Cam lower bound on resets is N* ∝ σ² / ||h_A - h_B||² ∝ (Δη)^-(4K-2).
# Pass: fitted slope of N* within ±0.3 of -(4K-2) for K = 2, 3, 4.
# The matching UPPER bound is checked separately (ESPRIT, cell T9) and must be proved.
# ============================================================================
L_T1, SIGMA_T1 = 400, 1.0
t1 = {}
for K, grid in [(2, np.geomspace(0.005, 0.08, 9)), (3, np.geomspace(0.01, 0.08, 8)), (4, np.geomspace(0.02, 0.08, 7))]:
    (sA, wA), (sB, wB), gap = matched_pair(K)
    dist, nstar = [], []
    for d in grid:
        rA, rB = R_CENTER + d * sA, R_CENTER + d * sB
        assert is_stable(W_TOTAL * wA, rA, A_DEF, KAPPA_DEF) and is_stable(W_TOTAL * wB, rB, A_DEF, KAPPA_DEF)
        hA = impulse(W_TOTAL * wA, rA, A_DEF, KAPPA_DEF, L_T1)
        hB = impulse(W_TOTAL * wB, rB, A_DEF, KAPPA_DEF, L_T1)
        D = np.linalg.norm(hA - hB)
        dist.append(D)
        nstar.append(SIGMA_T1 ** 2 * np.log(1 / (4 * 0.05)) / D ** 2)   # Le Cam / KL, 5% error
    slope = loglog_slope(grid[:5], nstar[:5])          # asymptotic (small-Δη) regime
    t1[K] = {"delta_grid": grid, "response_gap": dist, "N_star_lower": nstar,
             "slope_N": slope, "target": -(4 * K - 2), "moment_gap_order_2K-1": gap}
    print(f"K={K}: slope of N* = {slope:6.2f}  (target {-(4*K-2)})")
record("T1_K_cohort_rate", all(abs(v["slope_N"] - v["target"]) <= 0.3 for v in t1.values()),
       "theorem-check", t1, criterion="slope within ±0.3 of -(4K-2), K=2,3,4",
       note="Lower bound only. Upper bound: see T9 (ESPRIT) and a written proof.")


In [ ]:
# ============================================================================
# CELL T2 — PASSIVE IDENTIFICATION (what passive price data can and cannot do)
# (a) Key lemma: the zeros of H(z) are the memory rates r_i, so with |r_i| < 1
#     H is minimum-phase and a single-shock price spectrum identifies H, hence
#     the merged measure. Check on 2,000 random stable economies, including
#     signed (mean-reverting) cohorts.
# (b) Exploratory: when each cohort's memory also receives its own shock, does
#     the price spectrum still pin down the parameters? Rank of the Jacobian.
# NOTE: this corrects the plan's first draft of T2. Signed cohorts do NOT break
# minimum phase. Passive noise generally HELPS identification (consistent with
# Paper 1 Prop 2.10); the barrier is the COST (T3: same δ^-4 / Δη^-6 orders).
# ============================================================================
def H_poly(w, r, a, kappa):
    """H(z) = kappa*D(z) / ((z-a)D(z) - kappa*N(z)), with D = prod(z-r_i), N = sum w_i z prod_{j!=i}(z-r_j)."""
    D = np.poly(r)
    N = np.zeros(len(r) + 1)
    for i in range(len(r)):
        N += w[i] * np.concatenate([np.atleast_1d(np.poly(np.delete(r, i))), [0.0]])
    den = np.polysub(np.polymul([1.0, -a], D), kappa * N)
    return kappa * D, den

R2 = rng(2)
n_ok, n_tot, worst = 0, 0, 0.0
for _ in range(2000):
    K = R2.integers(1, 5)
    r = R2.uniform(0.05, 0.95, K)
    w = R2.uniform(-0.15, 0.15, K)
    if not is_stable(w, r, A_DEF, KAPPA_DEF):
        continue
    num, den = H_poly(w, r, A_DEF, KAPPA_DEF)
    zeros = np.roots(num)
    n_tot += 1
    match = np.max(np.abs(np.sort(zeros) - np.sort(r))) < 1e-8
    inside = np.all(np.abs(zeros) < 1) and np.all(np.abs(np.roots(den)) < 1)
    n_ok += bool(match and inside)
print(f"(a) zeros of H equal memory rates and lie inside the unit disk: {n_ok}/{n_tot}")
record("T2a_min_phase", n_ok == n_tot, "theorem-check",
       {"n_ok": n_ok, "n_tested": n_tot},
       criterion="all stable economies (signed weights allowed) are minimum-phase with zeros = r_i")

# (b) two shock sources: price noise w_t and cohort memory noise e_{i,t}
def price_spectrum_2shock(theta, omegas, a=A_DEF, kappa=KAPPA_DEF):
    q1, q2, eta1, eta2, sw, se = theta
    S = np.zeros_like(omegas)
    for j, om in enumerate(omegas):
        z = np.exp(1j * om)
        r = np.array([1 - eta1, 1 - eta2]); q = np.array([q1, q2]); eta = np.array([eta1, eta2])
        # state equations in z: (1 - a z^-1) X = kappa z^-1 sum q_i Y_i + Wn
        #                         (1 - r_i z^-1) Y_i = eta_i X + E_i
        M = np.zeros((3, 3), complex)
        M[0, 0] = 1 - a / z; M[0, 1:] = -kappa * q / z
        for i in range(2):
            M[1 + i, 0] = -eta[i]; M[1 + i, 1 + i] = 1 - r[i] / z
        Minv = np.linalg.inv(M)
        S[j] = sw ** 2 * abs(Minv[0, 0]) ** 2 + se ** 2 * (abs(Minv[0, 1]) ** 2 + abs(Minv[0, 2]) ** 2)
    return S

omg = np.linspace(0.01, np.pi, 300)
theta0 = np.array([0.12, 0.10, 0.20, 0.45, 1.0, 0.3])
eps = 1e-6
Jac = np.column_stack([(np.log(price_spectrum_2shock(theta0 + eps * e, omg)) -
                        np.log(price_spectrum_2shock(theta0 - eps * e, omg))) / (2 * eps)
                       for e in np.eye(len(theta0))])
sv = np.linalg.svd(Jac, compute_uv=False)
rank = int(np.sum(sv > sv[0] * 1e-8))
print(f"(b) two-shock spectrum: Jacobian rank {rank} of {len(theta0)} parameters; singular values {np.round(sv, 6)}")
record("T2b_two_shock_rank", None, "theorem-check",
       {"rank": rank, "n_params": len(theta0), "singular_values": sv},
       note="Rank < 6 => local non-identification from passive prices; rank = 6 => identified locally.")


In [ ]:
# ============================================================================
# CELL T3 — PASSIVE SAMPLE COMPLEXITY via Whittle KL
# T* ∝ 1 / KL rate. Pass: slope -4 (symmetric pair, δ) and -6 (K=2 matched pair).
# ============================================================================
def spec_single_shock(w, r, omegas, a=A_DEF, kappa=KAPPA_DEF):
    num, den = H_poly(np.asarray(w), np.asarray(r), a, kappa)
    z = np.exp(1j * omegas)
    return np.abs(np.polyval(num, z) / np.polyval(den, z)) ** 2

def whittle_kl(S0, S1, omegas):
    ratio = S1 / S0
    return np.trapezoid(ratio - 1 - np.log(ratio), omegas) / (2 * np.pi)

omg3 = np.linspace(1e-4, np.pi, 20001)
grid_d = np.geomspace(0.01, 0.12, 9)
kl_pair, kl_k2 = [], []
(sA2, wA2), (sB2, wB2), _ = matched_pair(2)
for d in grid_d:
    S0 = spec_single_shock([W_TOTAL], [R_CENTER], omg3)
    S1 = spec_single_shock([W_TOTAL / 2, W_TOTAL / 2], [R_CENTER - d, R_CENTER + d], omg3)
    kl_pair.append(whittle_kl(S0, S1, omg3))
    SA = spec_single_shock(W_TOTAL * wA2, R_CENTER + d * sA2, omg3)
    SB = spec_single_shock(W_TOTAL * wB2, R_CENTER + d * sB2, omg3)
    kl_k2.append(whittle_kl(SA, SB, omg3))
s_pair, s_k2 = loglog_slope(grid_d, 1 / np.array(kl_pair)), loglog_slope(grid_d, 1 / np.array(kl_k2))
print(f"T* slope: pair {s_pair:.2f} (target -4), K=2 matched {s_k2:.2f} (target -6)")
record("T3_passive_rate", abs(s_pair + 4) <= 0.3 and abs(s_k2 + 6) <= 0.3, "theorem-check",
       {"delta": grid_d, "kl_pair": kl_pair, "kl_K2": kl_k2, "slope_pair": s_pair, "slope_K2": s_k2},
       criterion="slopes -4 and -6 within ±0.3")


In [ ]:
# ============================================================================
# CELL T4 — DISCLOSURE: cohort-tagged flow removes the separation barrier
# Regulator sees each cohort's flow u_{i,t} = q_i y_{i,t-1} with noise.
# Pass: RMSE of recovered weights is flat in Δη (|slope| <= 0.3).
# ============================================================================
def tagged_estimate(w, r, T, sig_u, R, a=A_DEF, kappa=KAPPA_DEF, sig_x=1.0):
    K = len(w); eta = 1 - r; q = w / eta
    x = np.zeros(T); y = np.zeros((T, K))
    shocks = R.normal(0, sig_x, T)
    for t in range(1, T):
        x[t] = a * x[t - 1] + kappa * np.dot(q, y[t - 1]) + shocks[t]
        y[t] = r * y[t - 1] + eta * x[t]
    u = q[None, :] * y + R.normal(0, sig_u, (T, K))     # observed tagged flow
    est = []
    for i in range(K):  # u_{i,t} = r_i u_{i,t-1} + w_i x_t  (noise -> use IV: x_t, u_{i,t-2})
        Y = u[2:, i]; X = np.column_stack([u[1:-1, i], x[2:]]); Z = np.column_stack([u[:-2, i], x[2:]])
        b = np.linalg.solve(Z.T @ X, Z.T @ Y)
        est.append(b[1])
    return np.array(est)

R4 = rng(4)
rmse4 = []
grid4 = np.geomspace(0.01, 0.12, 6)
for d in grid4:
    w_true = np.array([W_TOTAL / 2, W_TOTAL / 2]); r_true = np.array([R_CENTER - d, R_CENTER + d])
    errs = [tagged_estimate(w_true, r_true, 5000, 0.05, R4) - w_true for _ in range(60)]
    rmse4.append(float(np.sqrt(np.mean(np.square(errs)))))
s4 = loglog_slope(grid4, rmse4)
print(f"tagged-flow weight RMSE by Δη: {np.round(rmse4, 5)}  slope {s4:.2f}")
record("T4_disclosure", abs(s4) <= 0.3, "theorem-check",
       {"delta": grid4, "rmse": rmse4, "slope": s4, "T": 5000, "reps": 60},
       criterion="weight error flat in Δη (|slope| <= 0.3)")


In [ ]:
# ============================================================================
# CELL T5 — SPECTRAL BLINDNESS
# Markets with the same eigenvalues, the same one-step price volatility and the
# same stationary price variance, but worst-case transient loss growing without bound.
# ============================================================================
lam1, lam2, s1, V = 0.80, 0.70, 1.0, 5.0
rows5 = []
for th in [1.0, 3.0, 10.0, 30.0, 100.0]:
    J = np.array([[lam1, th], [0.0, lam2]])
    def var_price(s2):
        P = linalg.solve_discrete_lyapunov(J, np.diag([s1, s2]))
        return P[0, 0]
    base = var_price(0.0)
    s2 = (V - base) / (var_price(1.0) - base)   # variance is affine in s2
    P = linalg.solve_discrete_lyapunov(J, np.diag([s1, s2]))
    powers = [np.linalg.matrix_power(J, k) for k in range(200)]
    peak_state = max(np.linalg.norm(Pk[0, :]) for Pk in powers)          # worst unit state shock
    cum_state = sum(np.linalg.norm(Pk[0, :]) for Pk in powers)
    rows5.append({"theta": th, "eigs": np.sort(np.linalg.eigvals(J).real), "one_step_vol": s1,
                  "stationary_var": P[0, 0], "peak_gain": peak_state, "cum_gain": cum_state})
df5 = pd.DataFrame(rows5)
print(df5[["theta", "stationary_var", "peak_gain", "cum_gain"]].to_string(index=False))
same_var = np.allclose(df5["stationary_var"], V, atol=1e-8)
grows = np.all(np.diff(df5["peak_gain"]) > 0)
record("T5_spectral_blindness", bool(same_var and grows), "theorem-check", df5.to_dict("list"),
       criterion="identical eigenvalues, one-step vol and stationary variance; peak gain strictly increasing")


In [ ]:
# ============================================================================
# CELL L26 — PAPER 1 LEMMA 2.6: positive cohort economies cannot amplify
# Check: for stable economies with all w_i > 0, the price impulse response is
# completely monotone: (-1)^n Δ^n h_k >= 0 for n = 1..4.  (R2-T4)
# ============================================================================
R26 = rng(26)
viol, tested = 0, 0
for _ in range(2000):
    K = R26.integers(1, 5)
    r = R26.uniform(0.05, 0.95, K)
    w = R26.uniform(0.0, 0.2, K)
    a = R26.uniform(0.0, 0.95)
    if not is_stable(w, r, a, KAPPA_DEF):
        continue
    h = impulse(w, r, a, KAPPA_DEF, 200)
    tested += 1
    ok = all(np.all(((-1) ** n) * np.diff(h, n) >= -1e-12) for n in range(1, 5))
    viol += (not ok)
print(f"completely monotone: {tested - viol}/{tested}")
record("L26_complete_monotonicity", viol == 0, "theorem-check", {"tested": tested, "violations": viol},
       criterion="(-1)^n Δ^n h >= 0, n=1..4, every stable positive economy")


In [ ]:
# ============================================================================
# CELL T6 — THE AMPLIFICATION THEOREM (depth-depletion channel)
# Book model (state: price displacement x, signed depth imbalance d):
#   x_t = a x_{t-1} + phi d_{t-1} + kappa u_t        (book pressure moves price)
#   d_t = (1 - r) d_{t-1} + eps u_t                  (sweeps deplete one side; refill r)
# Theorem to check:  max_k h_k > h_0   <=>   phi*eps > (1 - a)*kappa.
# Peak time from h_k = A a^k + B b^k, b = 1 - r.  Refill r controls height/duration.
# With momentum flow f_t = (1-eta) f_{t-1} + eta q x_{t-1} added to u (3-state):
#   candidate condition  phi*eps + kappa^2*eta*q > (1 - a)*kappa  (reported, not assumed).
# ============================================================================
def book_impulse(a, phi, kappa, r, eps, eta=0.0, q=0.0, L=400):
    x = np.zeros(L); d = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        u = (1.0 if t == 0 else 0.0)
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = u + f[t]
        x[t] = (a * x[t - 1] if t else 0) + (phi * d[t - 1] if t else 0) + kappa * tot
        d[t] = ((1 - r) * d[t - 1] if t else 0) + eps * tot
    return x

R6 = rng(6)
agree2, n2, peak_ok, n_pk, mono_r, n_r = 0, 0, 0, 0, 0, 0
for _ in range(2000):
    a, r = R6.uniform(0.0, 0.97), R6.uniform(0.01, 0.99)
    kappa, phi, eps = R6.uniform(0.1, 2.0), R6.uniform(0.0, 1.0), R6.uniform(0.0, 1.0)
    if abs(a - (1 - r)) < 1e-3:
        continue
    h = book_impulse(a, phi, kappa, r, eps)
    amp_sim = h.max() > h[0] * (1 + 1e-12)
    amp_thm = phi * eps > (1 - a) * kappa
    n2 += 1; agree2 += (amp_sim == amp_thm)
    if amp_thm:
        b = 1 - r
        B = -phi * eps / (a - b); A = kappa - B
        if A * np.log(a) != 0 and (-B * np.log(b)) / (A * np.log(a)) > 0:
            tc = np.log((-B * np.log(b)) / (A * np.log(a))) / np.log(a / b)
            kpk = int(np.argmax(h))
            n_pk += 1; peak_ok += (np.floor(tc) <= kpk <= np.ceil(tc) + 1e-9)
        # comparative static: faster refill -> lower peak
        h2 = book_impulse(a, phi, kappa, min(r * 1.05, 0.999), eps)
        n_r += 1; mono_r += (h2.max() <= h.max() + 1e-12)
print(f"2-state iff condition agrees with simulation: {agree2}/{n2}")
print(f"closed-form peak time correct: {peak_ok}/{n_pk};  faster refill lowers the peak: {mono_r}/{n_r}")

agree3, n3 = 0, 0
for _ in range(2000):
    a, r = R6.uniform(0.0, 0.95), R6.uniform(0.05, 0.95)
    kappa, phi, eps = R6.uniform(0.1, 1.5), R6.uniform(0.0, 0.6), R6.uniform(0.0, 0.6)
    eta, q = R6.uniform(0.05, 0.95), R6.uniform(0.0, 0.8)
    J3 = np.array([[a, phi, kappa * (1 - eta)], [0, 1 - r, eps * (1 - eta)], [eta * q, 0, 1 - eta]])
    # (exact 3x3 transition is computed by simulation; stability checked from the path)
    h = book_impulse(a, phi, kappa, r, eps, eta, q, L=600)
    if not np.all(np.isfinite(h)) or abs(h[-1]) > 1e-6 * abs(h).max():
        continue
    n3 += 1
    agree3 += ((h.max() > h[0] * (1 + 1e-12)) == (phi * eps + kappa ** 2 * eta * q > (1 - a) * kappa))
print(f"3-state candidate condition agrees with simulation: {agree3}/{n3}")
record("T6_amplification_theorem", agree2 == n2 and peak_ok == n_pk and mono_r == n_r, "theorem-check",
       {"agree_2state": agree2, "n_2state": n2, "peak_time_ok": peak_ok, "n_peak": n_pk,
        "refill_monotone": mono_r, "n_refill": n_r, "agree_3state_candidate": agree3, "n_3state": n3},
       criterion="2-state: iff condition, peak time and refill comparative static hold in every draw",
       note="3-state agreement is reported: if < 100%, the 3-state theorem needs a sharper condition.")


In [ ]:
# ============================================================================
# CELL P1.7 — THIRD-TAP UNIT TEST (code correctness, belongs in an appendix)
# Merged cohort (weight q at rate η̄) vs split cohorts (q/2 at η̄ ± δ):
# first differing response lag and its size vs κ² q δ².
# ============================================================================
a7, k7, q7, eb = A_DEF, KAPPA_DEF, 0.3, 0.30
rows7 = []
for dl in [0.02, 0.05, 0.10]:
    hm = impulse([q7 * eb], [1 - eb], a7, k7, 20)
    hs = impulse([q7 / 2 * (eb + dl), q7 / 2 * (eb - dl)], [1 - eb - dl, 1 - eb + dl], a7, k7, 20)
    diff = hs - hm
    lag = int(np.argmax(np.abs(diff) > 1e-14))
    rows7.append({"delta": dl, "first_lag": lag, "diff": diff[lag], "kappa2_q_delta2": k7 ** 2 * q7 * dl ** 2})
df7 = pd.DataFrame(rows7); print(df7.to_string(index=False))
ok7 = np.allclose(np.abs(df7["diff"]), df7["kappa2_q_delta2"], rtol=1e-10)
record("P1.7-T1_third_tap", bool(ok7), "simulation", df7.to_dict("list"),
       criterion="|first response difference| = κ² q δ² to 1e-10 (unit test, not evidence)")


In [ ]:
# ============================================================================
# CELL T7 + P2.11 — MAXIMALITY OF THE GUARD, AND MONOTONICITY OF L_max(α)
# Price impact system J(α) = J0 + α B; per-step shock |u_t| <= ū; horizon H.
# Exact worst-case loss: |I(α)| ū Σ_{k<H} |C J(α)^k b|  (cumulative L1 gain).
# Check 1 (T7): the L1 guard admits exactly the largest α a brute-force adversary allows.
# Check 2: a peak-gain guard (|I| ū max_k|g_k|) can be UNSAFE; a peak×H guard is conservative.
# Check 3 (P2.11): is worst-case loss monotone in α on the grid?
# ============================================================================
H7, GRID7 = 12, np.round(np.arange(0, 1.0001, 0.025), 3)
C7 = np.array([1.0, 0, 0]); b7 = np.array([0.8, 0.32, 0.25])
signs = np.array(list(itertools.product([-1.0, 1.0], repeat=H7)))      # 4096 x H

def gains(J, H=H7):
    v, out = b7.copy(), []
    for _ in range(H):
        out.append(C7 @ v); v = J @ v
    return np.array(out)

R7 = rng(7)
res7 = {"brute_equal": 0, "n_brute": 0, "peak_unsafe": 0, "peakH_conservative": 0, "n": 0, "monotone_viol": 0}
for s in range(2000):
    J0 = np.diag(R7.uniform(0.5, 0.85, 3)) + np.triu(R7.uniform(0, 0.4, (3, 3)), 1)
    Bm = np.triu(R7.uniform(0, 0.6, (3, 3)), 1)
    I0, cf, ubar = R7.uniform(0.5, 2.0), R7.uniform(0.0, 2.0), R7.uniform(0.1, 1.0)
    G = np.array([gains(J0 + al * Bm) for al in GRID7])                  # (41, H)
    inv = np.abs(I0 + cf * GRID7)
    loss_L1 = inv * ubar * np.abs(G).sum(1)
    budget = np.quantile(loss_L1, R7.uniform(0.2, 0.9))
    res7["n"] += 1
    res7["monotone_viol"] += int(np.any(np.diff(loss_L1) < -1e-12))
    safe_L1 = GRID7[loss_L1 <= budget]
    a_L1 = safe_L1.max() if len(safe_L1) else -1
    loss_pk = inv * ubar * np.abs(G).max(1)
    a_pk = GRID7[loss_pk <= budget].max() if np.any(loss_pk <= budget) else -1
    loss_pkH = loss_pk * H7
    a_pkH = GRID7[loss_pkH <= budget].max() if np.any(loss_pkH <= budget) else -1
    if a_pk > a_L1:
        res7["peak_unsafe"] += 1
    if a_pkH < a_L1:
        res7["peakH_conservative"] += 1
    if s < 200:   # brute force adversary over all sign sequences and all stopping times
        true_worst = []
        for gi, g in zip(range(len(GRID7)), G):
            # displacement after n steps with shocks u_0..u_{n-1}: sum_t g_{n-1-t} u_t
            paths = np.stack([signs[:, :n] @ g[:n][::-1] for n in range(1, H7 + 1)], 1)
            true_worst.append(inv[gi] * ubar * np.abs(paths).max())
        true_worst = np.array(true_worst)
        safe_b = GRID7[true_worst <= budget * (1 + 1e-12)]
        a_b = safe_b.max() if len(safe_b) else -1
        res7["n_brute"] += 1; res7["brute_equal"] += int(np.isclose(a_b, a_L1))
print(json.dumps(res7))
record("T7_guard_maximality", res7["brute_equal"] == res7["n_brute"], "theorem-check", res7,
       criterion="L1-gain guard α equals brute-force maximal safe α in every state",
       note=f"peak-gain guard admitted an unsafe α in {res7['peak_unsafe']}/{res7['n']} states; "
            f"peak×H guard was conservative in {res7['peakH_conservative']}/{res7['n']}.")
record("P2.11-T1_monotone_Lmax", res7["monotone_viol"] == 0, "theorem-check",
       {"violations": res7["monotone_viol"], "states": res7["n"]},
       criterion="worst-case loss nondecreasing in α on the grid (else condition Theorem 2.12)")


In [ ]:
# ============================================================================
# CELL T8 — CERTIFIED ROBUST GAIN
# If ||E|| <= ε and ε S < 1 with M = sup_k ||J^k||, S = Σ_k ||J^k||, then
#   sup_k ||(J+E)^k|| <= M / (1 - ε S).
# Check on 100 random non-normal J × 300 perturbations (random + adversarial rank-1).
# ============================================================================
R8 = rng(8)
n8, viol8, ratios = 0, 0, []
for _ in range(100):
    d = R8.integers(2, 6)
    T = np.diag(R8.uniform(0.3, 0.85, d)) + np.triu(R8.normal(0, 1.5, (d, d)), 1)
    Q, _ = np.linalg.qr(R8.normal(size=(d, d)))
    J = Q @ T @ Q.T
    pn = power_norms(J)
    M, S = pn.max(), pn.sum()
    eps8 = 0.5 / S
    bound = M / (1 - eps8 * S)
    for j in range(300):
        if j % 2 == 0:
            E = R8.normal(size=(d, d))
        else:   # adversarial direction: align with top singular vectors of a high power of J
            k = R8.integers(1, 30)
            U, _, Vt = np.linalg.svd(np.linalg.matrix_power(J, k))
            E = np.outer(Vt[0], U[:, 0])
        E *= eps8 / np.linalg.norm(E, 2)
        m = power_norms(J + E, K=1500).max()
        n8 += 1; viol8 += (m > bound * (1 + 1e-9)); ratios.append(m / bound)
print(f"bound held in {n8 - viol8}/{n8}; tightness (realized/bound): median {np.median(ratios):.3f}, max {np.max(ratios):.3f}")
record("T8_robust_gain", viol8 == 0, "theorem-check",
       {"n": n8, "violations": viol8, "tightness_median": np.median(ratios), "tightness_max": np.max(ratios)},
       criterion="realized sup_k ||(J+E)^k|| <= M/(1-εS) for every draw")


In [ ]:
# ============================================================================
# CELL T9 + P1.2-T1 — THE BRIDGE: wedge is cheap, cohorts are expensive
# K=2 cohorts at rates 0.7 ± Δ. Observe ĥ = h + noise (σ_eff = σ/√N).
#   * transient wedge W = Σ_{k<L} |h_k|  -> relative error vs Δ (should be flat)
#   * cohort weights by deconvolution + ESPRIT -> error vs Δ (should blow up)
#   * local ESPRIT requirement N*_ESPRIT(Δ) ∝ Δ^-6 (upper-bound side of P1.2)
# ============================================================================
def esprit_weights(k, K=2, M=None):
    M = M or (len(k) - 1) // 2
    Hk = linalg.hankel(k[:M + 1], k[M:2 * M + 1])
    U, s, Vt = np.linalg.svd(Hk)
    Us = U[:, :K]
    z = np.linalg.eigvals(np.linalg.pinv(Us[:-1]) @ Us[1:])
    z = np.sort(np.real(z))
    V = np.vander(z, len(k), increasing=True).T
    wv = np.linalg.lstsq(V, k, rcond=None)[0]
    return z, np.real(wv)

L9, sig_eff = 60, 1e-6
R9 = rng(9)
grid9 = np.geomspace(0.03, 0.2, 7)
rows9 = []
for dlt in grid9:
    w_t = np.array([0.05, 0.05]); r_t = np.array([R_CENTER - dlt, R_CENTER + dlt])
    h = impulse(w_t, r_t, A_DEF, KAPPA_DEF, L9)
    W_true = np.abs(h).sum()
    we, ce = [], []
    for _ in range(300):
        hh = h + R9.normal(0, sig_eff, L9)
        we.append(abs(np.abs(hh).sum() - W_true) / W_true)
        kk = kernel_from_impulse(hh, A_DEF, KAPPA_DEF)[:24]
        z, wv = esprit_weights(kk)
        ce.append(np.max(np.abs(wv - w_t)))
    q95 = np.quantile(ce, 0.95)
    rows9.append({"delta": dlt, "wedge_rel_err_q95": np.quantile(we, 0.95), "weight_err_q95": q95,
                  "N_star_ESPRIT_tau0.01": (q95 / sig_eff * 1.0 / 0.01) ** 2})   # σ=1 per reset
df9 = pd.DataFrame(rows9); print(df9.to_string(index=False))
s_w = loglog_slope(df9["delta"][:4], df9["wedge_rel_err_q95"][:4])     # small-Δ regime (the theorem's limit)
s_c = loglog_slope(df9["delta"][:4], df9["weight_err_q95"][:4])
s_N = loglog_slope(df9["delta"][:4], df9["N_star_ESPRIT_tau0.01"][:4])
print(f"slopes: wedge error {s_w:.2f} (target 0), weight error {s_c:.2f} (target -3), N*_ESPRIT {s_N:.2f} (target -6)")
record("T9_bridge", abs(s_w) <= 0.3 and s_c <= -2.5, "theorem-check",
       {"table": df9.to_dict("list"), "slope_wedge": s_w, "slope_weights": s_c},
       criterion="as Δ -> 0 (4 smallest Δ): wedge error flat (|slope| <= 0.3); cohort-weight error grows at least like Δ^-2.5")
record("P1.2-T1_esprit_rate", abs(s_N + 6) <= 0.5, "theorem-check",
       {"slope_N_star_ESPRIT": s_N, "table": df9.to_dict("list")},
       criterion="local ESPRIT requirement slope within ±0.5 of -6 (matches Le Cam order)",
       note="Numerical evidence for the upper bound. The paper still needs a written proof (Route A) "
            "or must call this a numerical proposition (Route B).")


## P1 / P2. Estimator and controller properties (simulation)

In [ ]:
# ============================================================================
# CELL P1.5 — FEEDBACK ESTIMATORS UNDER ORDER SPLITTING (simulation, 500 reps)
# Simulated market (all parameters known):
#   y_t  = (1-η) y_{t-1} + η Δp_{t-1}        algorithm memory of past returns
#   u_t  = ρ u_{t-1} + ν_t                   persistent order-splitting flow
#   q_t  = γ y_t + u_t                        signed flow
#   Δp_t = κ q_t + λ z_t + e_t                z_t: observed exogenous price shock
# Instruments for y_t: z_{t-1}, z_{t-2}, z_{t-3} (move y through Δp, independent of u).
# Tests: size at γ=0 (P1.5-T1), 95% coverage at γ=0.2 (P1.5-T2), placebo instrument (P1.5-T3).
# ============================================================================
def nw_cov(Xh, e, lags):
    """Newey-West long-run covariance of Xh*e."""
    g = Xh * e[:, None]
    S = g.T @ g
    for l in range(1, lags + 1):
        w = 1 - l / (lags + 1)
        G = g[l:].T @ g[:-l]
        S += w * (G + G.T)
    return S

def _proj(Z, M):
    """P_Z M without forming the n x n projection matrix."""
    return Z @ np.linalg.solve(Z.T @ Z, Z.T @ M)

def iv_fit(y, X, Z, kind="2sls", lags=10):
    """X: regressors (incl. const), Z: instruments (incl. const). Returns beta, HAC se."""
    if kind == "ols":
        Xh = X
    elif kind == "2sls":
        Xh = _proj(Z, X)
    elif kind == "liml":
        Yall = np.column_stack([y, X[:, 1:]])
        Yd = Yall - Yall.mean(0)                       # M_exog with exog = constant
        A = Yd.T @ Yd
        Bm = Yall.T @ Yall - Yall.T @ _proj(Z, Yall)
        kk = np.min(np.real(linalg.eigvals(A, Bm)))
        Xh = X - kk * (X - _proj(Z, X))                # (I - k M_Z) X
    beta = np.linalg.solve(Xh.T @ X, Xh.T @ y)
    e = y - X @ beta
    Ainv = np.linalg.inv(Xh.T @ X)
    V = Ainv @ nw_cov(Xh, e, lags) @ Ainv.T
    return beta, np.sqrt(np.diag(V))

def simulate_market(T, gamma, R, eta=0.3, rho=0.8, kappa=0.5, lam=1.0):
    y = np.zeros(T); u = np.zeros(T); q = np.zeros(T); dp = np.zeros(T)
    z = R.normal(size=T); nu = R.normal(size=T); e = R.normal(size=T)
    for t in range(1, T):
        y[t] = (1 - eta) * y[t - 1] + eta * dp[t - 1]
        u[t] = rho * u[t - 1] + nu[t]
        q[t] = gamma * y[t] + u[t]
        dp[t] = kappa * q[t] + lam * z[t] + e[t]
    return y, q, z

def estimators(y, q, z, rho_oracle=0.8):
    s = slice(4, None)
    Y, X = q[s], np.column_stack([np.ones(len(q) - 4), y[s]])
    Z = np.column_stack([np.ones(len(q) - 4), z[3:-1], z[2:-2], z[1:-3]])
    out = {}
    out["OLS"] = iv_fit(Y, X, None, "ols")
    # Cochrane-Orcutt with oracle rho, and with estimated rho (iterated)
    for name, rh in [("CO_oracle", rho_oracle), ("CO_estimated", None)]:
        r_ = rh if rh is not None else 0.0
        for _ in range(1 if rh is not None else 20):
            Yq = q[5:] - r_ * q[4:-1]; Xq = np.column_stack([np.ones(len(Yq)), y[5:] - r_ * y[4:-1]])
            b = np.linalg.lstsq(Xq, Yq, rcond=None)[0]
            if rh is None:
                res = q[4:] - b[1] * y[4:]
                r_ = np.dot(res[1:], res[:-1]) / np.dot(res[:-1], res[:-1])
        out[name] = iv_fit(Yq, Xq, None, "ols")
    out["2SLS"] = iv_fit(Y, X, Z, "2sls")
    out["LIML"] = iv_fit(Y, X, Z, "liml")
    # first-stage F
    b1 = np.linalg.lstsq(Z, y[s], rcond=None)[0]
    rss1 = np.sum((y[s] - Z @ b1) ** 2); rss0 = np.sum((y[s] - y[s].mean()) ** 2)
    F = ((rss0 - rss1) / 3) / (rss1 / (len(Y) - 4))
    return out, F

T15, REPS15 = 3000, 500
R15 = rng(15)
cover, size, bias, Fs = {}, {}, {}, []
for rep in range(REPS15):
    for gam, store in [(0.2, cover), (0.0, size)]:
        y, q, z = simulate_market(T15, gam, R15)
        out, F = estimators(y, q, z)
        if gam == 0.2:
            Fs.append(F)
        for k, (b, se) in out.items():
            lo, hi = b[1] - 1.96 * se[1], b[1] + 1.96 * se[1]
            store.setdefault(k, []).append(lo <= gam <= hi)
            if gam == 0.2:
                bias.setdefault(k, []).append(b[1] - gam)
tab15 = pd.DataFrame({
    "coverage_at_0.2": {k: np.mean(v) for k, v in cover.items()},
    "rejection_at_0": {k: 1 - np.mean(v) for k, v in size.items()},
    "bias": {k: np.mean(v) for k, v in bias.items()},
    "rmse": {k: np.sqrt(np.mean(np.square(v))) for k, v in bias.items()},
})
tab15["label"] = np.where(tab15["coverage_at_0.2"].between(0.93, 0.97), "consistent", "biased / invalid inference")
print(tab15.round(4).to_string()); print(f"median first-stage F = {np.median(Fs):.1f}")
size_ok = tab15.loc[["2SLS", "LIML", "CO_oracle"], "rejection_at_0"].between(0.031, 0.069).all()
record("P1.5-T1_size", bool(size_ok), "simulation", tab15["rejection_at_0"].to_dict(),
       criterion="rejection rate at γ=0 in [3.1%, 6.9%] for estimators called consistent")
record("P1.5-T2_coverage", None, "simulation", {"table": tab15.to_dict(), "median_first_stage_F": np.median(Fs)},
       criterion="label every estimator outside [93%, 97%] coverage as biased in the paper's table",
       note="Use this table to decide which estimator the paper may call consistent.")

# Placebo instrument: future z (independent of y_t) should be weak and give nothing
y, q, z = simulate_market(T15, 0.2, rng(151))
s = slice(4, -6)
Zp = np.column_stack([np.ones(len(q[s])), z[9:-1], z[8:-2], z[7:-3]])
b1 = np.linalg.lstsq(Zp, y[s], rcond=None)[0]
rss1 = np.sum((y[s] - Zp @ b1) ** 2); rss0 = np.sum((y[s] - y[s].mean()) ** 2)
Fp = ((rss0 - rss1) / 3) / (rss1 / (len(y[s]) - 4))
record("P1.5-T3_placebo_instrument", bool(Fp < 10), "simulation", {"placebo_first_stage_F": Fp},
       criterion="placebo instrument first-stage F < 10")


In [ ]:
# ============================================================================
# CELL P2.4 — CONFIDENCE SET FOR THE OPERATOR AND CERTIFIED WEDGE (simulation)
# Window VAR(1) like the real-data estimator (n = 600). ε_t = 95th percentile of
# ||Ĵ* - Ĵ||₂ over a residual bootstrap. Coverage: true J inside the ball.
# Certificate: robust bound M/(1-εS) from T8, available when εS < 1.
# ============================================================================
def var1_ols(Z):
    X = np.column_stack([np.ones(len(Z) - 1), Z[:-1]])
    B = np.linalg.lstsq(X, Z[1:], rcond=None)[0]
    return B[1:].T, B[0], Z[1:] - X @ B

def sim_var(J, n, R, burn=200, Sig=None):
    d = len(J); Z = np.zeros((n + burn, d))
    L = np.linalg.cholesky(Sig) if Sig is not None else np.eye(d)
    for t in range(1, n + burn):
        Z[t] = J @ Z[t - 1] + L @ R.normal(size=d)
    return Z[burn:]

J_true = np.array([[0.90, 0.35], [0.02, 0.60]])
R24 = rng(24)
inside, cert = 0, 0
REPS24, B24, n24 = 300, 199, 600
eps_list = []

def var1_batch(Zb):
    """OLS VAR(1) with intercept for a batch of paths Zb (B, n, d) -> J (B, d, d)."""
    B_, n_, d_ = Zb.shape
    X = np.concatenate([np.ones((B_, n_ - 1, 1)), Zb[:, :-1]], 2)
    Y = Zb[:, 1:]
    XtX = np.einsum("bti,btj->bij", X, X); XtY = np.einsum("bti,btj->bij", X, Y)
    Bc = np.linalg.solve(XtX, XtY)
    return np.transpose(Bc[:, 1:], (0, 2, 1))

for rep in range(REPS24):
    Z = sim_var(J_true, n24, R24)
    Jh, c, res = var1_ols(Z)
    idx = R24.integers(0, len(res), (B24, len(res)))
    Zb = np.zeros((B24, len(Z), 2)); Zb[:, 0] = Z[0]
    for t in range(1, len(Z)):
        Zb[:, t] = c + Zb[:, t - 1] @ Jh.T + res[idx[:, t - 1]]
    Jb = var1_batch(Zb)
    dists = np.linalg.norm(Jb - Jh[None], 2, axis=(1, 2))
    eps_t = np.quantile(dists, 0.95); eps_list.append(eps_t)
    inside += np.linalg.norm(Jh - J_true, 2) <= eps_t
    pn = power_norms(Jh, 2000)
    cert += eps_t * pn.sum() < 1
cov24 = inside / REPS24
print(f"bootstrap ball coverage = {cov24:.3f}; certificate available in {cert}/{REPS24} windows; median ε = {np.median(eps_list):.4f}")
record("P2.4-T1_coverage", bool(0.925 <= cov24 <= 0.975), "simulation",
       {"coverage": cov24, "reps": REPS24, "boot": B24, "median_eps": np.median(eps_list),
        "certificate_available_share": cert / REPS24},
       criterion="true J inside the 95% ball in [92.5%, 97.5%] of reps",
       note="If coverage is low, the paper must call ε_t approximate or use a larger ball.")


## P2 / P1. Real-data tests (Binance 1-second klines, checksum-verified)
These are the tests that decide gates **G1** (artifact null) and **G2** (predictive power). Whatever they show goes in the paper.

In [ ]:
# ============================================================================
# CELL P2.2 — UNIT-INVARIANT AMPLIFICATION ON REAL CRASHES + ARTIFACT NULL   (gate G1)
# Estimator (per 10-minute window, 1-second bars), following the repo's pipeline:
#   x_t = log-price deviation from its EMA (bp), f_t = next-second signed taker flow ($1M)
#   VAR(1) with intercept on z_t = (x_t, f_t)  ->  Ĵ (2x2)
# Statistics (all dimensionless and invariant to the units of x and f):
#   M_std = max_k || e_xᵀ J̃^k ||₂ with J̃ = D⁻¹ Ĵ D, D = diag(sd x, sd f)   <- PRIMARY
#           (the paper's M_C made unit-free: the state is measured in standard deviations)
#   A_xx  = max_k |(Ĵ^k)_{11}|      price-to-price amplification (reported)
#   hump  = max_k |(Ĵ^k)_{12}| / |Ĵ_{12}|   delayed peak of price response to a flow shock (reported)
#   rho   = spectral radius of Ĵ
# Window sets:
#   "stress_ex_ante": windows whose PREVIOUS 10 minutes had top-decile realized vol (no outcome selection)
#   "all_crash_day":  every window on the crash day
#   "ordinary":       every window on panel days in 2021-2023 (placebo)
#   "outcome_selected": top-10 |return| windows per crash day (old method; shown for comparison only)
# Artifact null: keep each window's REAL flow, generate price with NO feedback
#   (Δp = κ̂ f + σ̂ e), push through the identical EMA + VAR pipeline.
# ============================================================================
WIN, SPAN, KMAX = 600, 120, 300

def var1_J(x, f):
    """OLS VAR(1) with intercept. Returns NaNs for degenerate windows (no trades / flat price)."""
    Z = np.column_stack([x, f])
    if not np.all(np.isfinite(Z)) or Z[:, 0].std() < 1e-9 or Z[:, 1].std() < 1e-12:
        return np.full((2, 2), np.nan)
    X = np.column_stack([np.ones(len(Z) - 1), Z[:-1]])
    B = np.linalg.lstsq(X, Z[1:], rcond=None)[0]
    return B[1:].T

STAT = "M_std"          # primary statistic used by the gate tests

def _powers_rows(J, kmax):
    """Rows e_xᵀ J^k for k = 0..kmax (fast eigen form, loop fallback), shape (kmax+1, 2)."""
    w, V = np.linalg.eig(J)
    if np.max(np.abs(w)) < 1.05 and np.linalg.cond(V) < 1e8:
        Vi = np.linalg.inv(V)
        k = np.arange(kmax + 1)[:, None]
        return np.real((w[None, :] ** k) @ (V[0, :][:, None] * Vi))
    rows, P = [], np.eye(2)
    for _ in range(kmax + 1):
        rows.append(P[0].copy()); P = P @ J
        if np.abs(P).max() > 1e8:
            break
    return np.array(rows)

def window_stats(J, sd_x, sd_f, kmax=KMAX):
    if not np.all(np.isfinite(J)) or sd_x <= 0 or sd_f <= 0:
        return {"M_std": np.nan, "A_xx": np.nan, "hump": np.nan, "rho": np.nan}
    D = np.diag([sd_x, sd_f]); Jt = np.linalg.solve(D, J @ D)
    R_raw = _powers_rows(J, kmax); R_std = _powers_rows(Jt, kmax)
    h = np.abs(R_raw[1:, 1])                     # (J^k)_{12}, k >= 1
    return {"M_std": float(np.linalg.norm(R_std, axis=1).max()),
            "A_xx": float(np.abs(R_raw[:, 0]).max()),
            "hump": float(h.max() / h[0]) if h[0] > 1e-15 else np.nan,
            "rho": float(np.max(np.abs(np.linalg.eigvals(J))))}

def amp_stats(J, kmax=KMAX):
    """(A_xx, rho) — kept for the predictive cell."""
    st = window_stats(J, 1.0, 1.0, kmax)
    return st["A_xx"], st["rho"]

def day_windows(day, span=SPAN, win=WIN, delta=1):
    p, f, v = bars(day, delta)
    x = ema_dev(p, span)
    f_next = np.roll(f, -1)
    n = len(p) // win
    out = []
    for i in range(1, n):            # skip first window (EMA burn-in)
        s = slice(i * win, (i + 1) * win - 1)
        prev = slice((i - 1) * win, i * win)
        st = window_stats(var1_J(x[s], f_next[s]), x[s].std(), f_next[s].std())
        st.update({"i": i, "rv_prev": float(np.std(np.diff(p[prev]))),
                   "abs_ret": float(abs(p[s][-1] - p[s][0]))})
        out.append(st)
    return pd.DataFrame(out).dropna(subset=[STAT]).reset_index(drop=True)

def null_window_stats(day, i, B, R, span=SPAN, win=WIN):
    """Artifact null for window i: real flow, no-feedback price, same pipeline."""
    p, f, _ = bars(day, 1)
    lo = max(0, i * win - 5 * span)
    seg = slice(lo, (i + 1) * win)
    dp = np.diff(p[seg]); ff = f[seg][1:]
    X = np.column_stack([np.ones(len(ff)), ff])
    beta, *_ = np.linalg.lstsq(X, dp, rcond=None)
    sig = np.std(dp - X @ beta)
    fn = np.roll(f[seg], -1)
    stats_ = []
    for _ in range(B):
        dps = beta[1] * ff + sig * R.normal(size=len(ff))
        ps = np.concatenate([[0.0], np.cumsum(dps)])
        xs = ema_dev(ps, span)
        w = slice(len(ps) - win, len(ps) - 1)
        st = window_stats(var1_J(xs[w], fn[w]), xs[w].std(), fn[w].std())
        stats_.append([st[STAT], st["hump"]])
    return np.array(stats_, float)          # (B, 2): primary statistic, hump

def boot_mean_ci(v, R, B=2000, block=3):
    v = np.asarray(v); n = len(v)
    nb = int(np.ceil(n / block))
    means = []
    for _ in range(B):
        starts = R.integers(0, max(1, n - block + 1), nb)
        idx = np.concatenate([np.arange(s, min(s + block, n)) for s in starts])[:n]
        means.append(v[idx].mean())
    return float(np.mean(v)), [float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))]

print("Downloading crash days and 2021-2023 panel days (cached after first run)...")
crash_data = {k: load_day("BTCUSDT", d) for k, d in {**CRASH_DAYS, **OOS_CRASH_DAYS}.items()}
ord_dates = [d for d in PANEL_DAYS if int(d[:4]) in TRAIN_YEARS]
ord_data = load_days("BTCUSDT", ord_dates)
assert len(ord_data) >= 0.9 * len(ord_dates), "too many panel days failed to download"

R22 = rng(22)
crash_w = {k: day_windows(v) for k, v in crash_data.items()}
ord_w = pd.concat([day_windows(v).assign(day=d) for d, v in ord_data.items()], ignore_index=True)
rv_cut = np.quantile(ord_w["rv_prev"], 0.90)          # ex-ante stress trigger, set on ordinary days

rows = []
def _row(name, sub):
    m, ci = boot_mean_ci(sub[STAT], R22)
    return {"set": name, "n_windows": len(sub), f"mean_{STAT}": m, "ci95": ci,
            "median_hump": sub["hump"].median(), "share_A_xx_gt_1": float(np.mean(sub["A_xx"] > 1 + 1e-9)),
            "mean_rho": sub["rho"].mean()}
rows.append(_row("ordinary (2021-23 panel)", ord_w))
null_rows = {}
for k, w in crash_w.items():
    stress = w[w["rv_prev"] >= rv_cut]
    for label, sub in [("all_crash_day", w), ("stress_ex_ante", stress),
                       ("outcome_selected(top10|ret|)", w.nlargest(10, "abs_ret"))]:
        if len(sub) < 3:
            continue
        rows.append(_row(f"{k}:{label}", sub))
    # artifact null on the ex-ante stress windows (or all windows if none qualify)
    target = stress if len(stress) >= 3 else w
    sims = np.array([null_window_stats(crash_data[k], int(i), 50, R22) for i in target["i"]])  # (n_w, 50, 2)
    null_means = np.nanmean(sims[:, :, 0], 0)            # null distribution of the window-mean (primary)
    null_hump = np.nanmedian(sims[:, :, 1], 0)           # null distribution of the window-median hump
    obs, obs_h = target[STAT].mean(), target["hump"].median()
    null_rows[k] = {"observed_mean": obs, "null_mean": float(null_means.mean()),
                    "null_p99": float(np.quantile(null_means, 0.99)),
                    "p_value": float((1 + np.sum(null_means >= obs)) / (1 + len(null_means))),
                    "share_windows_M_std_gt_1": float(np.mean(target[STAT] > 1 + 1e-9)),
                    "observed_median_hump": obs_h, "null_hump_p99": float(np.quantile(null_hump, 0.99)),
                    "hump_p_value": float((1 + np.sum(null_hump >= obs_h)) / (1 + len(null_hump))),
                    "n_windows": len(target)}

tab22 = pd.DataFrame(rows)
print(tab22.to_string(index=False))
print(pd.DataFrame(null_rows).T.to_string())
in_sample = [k for k in CRASH_DAYS]
g1 = all(null_rows[k]["observed_mean"] > null_rows[k]["null_p99"] for k in in_sample)
record("P2.2-T2_artifact_null_G1", g1, "real-data", {"table": tab22.to_dict("list"), "null": null_rows},
       criterion="crash-day stress mean M_std > 99th pct of the no-feedback pipeline null, all 4 events",
       note="If FAIL: write that empirical amplification is not distinguishable from pipeline artifacts. "
            "M_std = 1 means no transient growth at all in standardized units. Also read hump_p_value: "
            "a delayed price peak beyond the no-feedback null is the T6 signature.")

placebo_pass = sum(
    tab22.loc[tab22["set"] == f"{k}:stress_ex_ante", f"mean_{STAT}"].squeeze() > np.quantile(ord_w[STAT], 0.95)
    for k in in_sample if (tab22["set"] == f"{k}:stress_ex_ante").any())
record("P2.2-T3_calm_placebo", bool(placebo_pass >= 3), "real-data",
       {"events_above_ordinary_p95": int(placebo_pass), "ordinary_p95": np.quantile(ord_w[STAT], 0.95)},
       criterion="stress-window mean M_std above the ordinary-window 95th pct for >= 3 of 4 events")

sel = {k: {"ex_ante": tab22.loc[tab22["set"] == f"{k}:stress_ex_ante", f"mean_{STAT}"].tolist(),
           "outcome_selected": tab22.loc[tab22["set"] == f"{k}:outcome_selected(top10|ret|)", f"mean_{STAT}"].tolist()}
       for k in crash_w}
record("P2.2-T4_selection", None, "real-data", sel,
       note="Report both. If amplification appears only in outcome-selected windows, it is a selection effect.")

# ---- Unit invariance (P2.2-T1) --------------------------------------------
d0 = crash_data["May2021"]; p0, f0, _ = bars(d0); x0 = ema_dev(p0, SPAN); fn0 = np.roll(f0, -1)
s = slice(30000, 30599)
xa, fa = x0[s], fn0[s]; xb, fb_ = x0[s] * 10, fn0[s] * 1e-3
J1, J2 = var1_J(xa, fa), var1_J(xb, fb_)
w1, w2 = window_stats(J1, xa.std(), fa.std()), window_stats(J2, xb.std(), fb_.std())
st1 = float(np.linalg.norm(_powers_rows(J1, KMAX), axis=1).max())     # raw state norm (paper's M_C style)
st2 = float(np.linalg.norm(_powers_rows(J2, KMAX), axis=1).max())
inv_ok = all(abs(w1[k] - w2[k]) <= 1e-6 * max(1, abs(w1[k])) for k in ("M_std", "A_xx", "hump"))
record("P2.2-T1_unit_invariance", bool(inv_ok), "real-data",
       {"original": w1, "rescaled": w2, "raw_state_norm_original": st1, "raw_state_norm_rescaled": st2},
       criterion="M_std, A_xx, hump unchanged when price ×10 and flow ×0.001; raw state norm changes")

# ---- Sensitivity (P2.2.5) ---------------------------------------------------
sens = []
for span in (60, 120, 300):
    for delta in (1, 5):
        win = WIN // delta
        o = pd.concat([day_windows(v, max(2, span // delta), win, delta) for v in list(ord_data.values())[::6]])
        c = pd.concat([day_windows(v, max(2, span // delta), win, delta) for k, v in crash_data.items() if k in CRASH_DAYS])
        sens.append({"ema_span_s": span, "bar_s": delta, f"ordinary_mean_{STAT}": o[STAT].mean(),
                     f"crash_mean_{STAT}": c[STAT].mean(), "difference": c[STAT].mean() - o[STAT].mean()})
sens = pd.DataFrame(sens); print(sens.round(4).to_string(index=False))
record("P2.2-T5_sensitivity", None, "real-data", sens.to_dict("list"),
       note="The headline must hold (same sign) across EMA span and bar size; otherwise report the dependence.")


In [ ]:
# ============================================================================
# CELL T6-E — DOES PRICE OVERSHOOT AFTER A FLOW SHOCK? (real data, local projections)
# For horizons k: regress  p_{t+k} - p_{t-1}  (bp)  on  f_t ($1M signed flow),
# controls: 5 lags of returns and flows.  Two versions:
#   "total":           includes the effect of flow persistence (order splitting)
#   "flow_held_fixed": also controls for cumulative future flow f_{t+1..t+k}
# Overshoot (the T6 mechanism) = β_k > β_0 in the flow-held-fixed response.
# Day-clustered bootstrap CIs.  Run on 2021-2023 panel days and on crash days.
# ============================================================================
HORIZONS = [0, 1, 2, 5, 10, 20, 30, 60, 120, 300]

def lp_day(day, horizons=HORIZONS, nlags=5):
    p, f, _ = bars(day, 1)
    r = np.diff(p, prepend=p[0])
    n = len(p); maxk = max(horizons)
    t = np.arange(nlags + 1, n - maxk - 1)
    lagsR = np.column_stack([r[t - j] for j in range(1, nlags + 1)])
    lagsF = np.column_stack([f[t - j] for j in range(1, nlags + 1)])
    cumf = np.concatenate([[0.0], np.cumsum(f)])
    out = {"total": [], "flow_held_fixed": []}
    for k in horizons:
        y = p[t + k] - p[t - 1]
        Xb = np.column_stack([np.ones(len(t)), f[t], lagsR, lagsF])
        out["total"].append(np.linalg.lstsq(Xb, y, rcond=None)[0][1])
        fut = cumf[t + k + 1] - cumf[t + 1]              # f_{t+1} .. f_{t+k}
        Xf = np.column_stack([Xb, fut]) if k > 0 else Xb
        out["flow_held_fixed"].append(np.linalg.lstsq(Xf, y, rcond=None)[0][1])
    return {k: np.array(v) for k, v in out.items()}

def lp_summary(days, R, B=1000):
    per = [lp_day(d) for d in days.values()]
    res = {}
    for kind in ("total", "flow_held_fixed"):
        M = np.array([p[kind] for p in per])            # days x horizons
        mean = M.mean(0)
        boots = np.array([M[R.integers(0, len(M), len(M))].mean(0) for _ in range(B)])
        over = boots[:, 1:].max(1) - boots[:, 0]         # max_{k>=1} β_k - β_0 (can be negative)
        peak_k = HORIZONS[int(np.argmax(mean))]
        res[kind] = {"beta": mean, "ci_lo": np.quantile(boots, 0.025, 0), "ci_hi": np.quantile(boots, 0.975, 0),
                     "overshoot_mean": float(mean[1:].max() - mean[0]),
                     "overshoot_ci": [float(np.quantile(over, 0.025)), float(np.quantile(over, 0.975))],
                     "peak_horizon_s": peak_k,
                     "decays_after_peak": bool(mean[-1] < mean.max())}
    return res

R6E = rng(61)
lp_ord = lp_summary(ord_data, R6E)
lp_crash = lp_summary({k: v for k, v in crash_data.items()}, R6E, B=500)
for lab, res in [("ordinary", lp_ord), ("crash", lp_crash)]:
    for kind, v in res.items():
        print(f"{lab:9s} {kind:16s} β(bp per $1M) by horizon: {np.round(v['beta'], 3)}")
        print(f"{'':9s} {'':16s} overshoot {v['overshoot_mean']:.3f} CI {np.round(v['overshoot_ci'], 3)}, peak at {v['peak_horizon_s']}s")
record("T6-E_overshoot_real", None, "real-data",
       {"horizons_s": HORIZONS, "ordinary": lp_ord, "crash": lp_crash},
       note="Evidence for T6 = flow-held-fixed overshoot CI above 0, with a decay after the peak. "
            "A 'total' overshoot alone can come from order splitting and is not evidence for the depth channel.")


In [ ]:
# ============================================================================
# CELL P1.6 — IS MEASURED FEEDBACK MORE THAN EMA MECHANICS? (real data + placebo)
# Paper 1 estimator (repo feedback_var.var1): VAR(1) on (x_t, f_t) with EMA span 30 bins,
# Q = J12 J21 / (1 - J22).  Mechanical EMA persistence: 1 - 2/31 = 0.935.
# Placebo: same pipeline, price replaced by a random walk with the same volatility,
# independent of flow.  Pass: placebo Q CI contains 0 (pipeline unbiased).
# Report: whether real Q lies outside the placebo 95% band.
# ============================================================================
def q_stats(p, f, span=30):
    x = ema_dev(p, span); fn = np.roll(f, -1)
    x, fn = x[5 * span:-1], fn[5 * span:-1]
    J = var1_J(x, fn)
    Q = J[0, 1] * J[1, 0] / (1 - J[1, 1])
    return Q, J[0, 0]

R16 = rng(16)
real, plac = [], []
for d, day in ord_data.items():
    p, f, _ = bars(day, 1)
    real.append(q_stats(p, f))
    rw = np.concatenate([[p[0]], p[0] + np.cumsum(R16.normal(0, np.std(np.diff(p)), len(p) - 1))])
    plac.append(q_stats(rw, f))
real, plac = np.array(real), np.array(plac)
def day_ci(v, B=2000):
    bs = [v[R16.integers(0, len(v), len(v))].mean() for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]
res16 = {"Q_real_mean": real[:, 0].mean(), "Q_real_ci": day_ci(real[:, 0]),
         "Q_placebo_mean": plac[:, 0].mean(), "Q_placebo_ci": day_ci(plac[:, 0]),
         "Q_placebo_band95": [float(np.quantile(plac[:, 0], 0.025)), float(np.quantile(plac[:, 0], 0.975))],
         "J11_real_mean": real[:, 1].mean(), "J11_placebo_mean": plac[:, 1].mean(),
         "J11_mechanical": 1 - 2 / 31, "n_days": len(real)}
print(json.dumps(_clean(res16), indent=1))
ci = res16["Q_placebo_ci"]
record("P1.6-T1_ema_placebo", bool(ci[0] <= 0 <= ci[1]), "real-data", res16,
       criterion="placebo Q CI contains 0",
       note="Then report: real Q CI vs placebo band. If they overlap, write 'no evidence beyond mechanics'.")


In [ ]:
# ============================================================================
# CELL P1.3 — MEASUREMENT NOISE ON THE JAN 2023 HOLDOUT (real data)
# Roll-type estimator on 1-second closes: σ_noise = sqrt(-cov(Δp_t, Δp_{t-1})) in USDT.
# Capital scales as σ_obs²: C(σ_obs) = 61.44 USDT × (σ_obs / 0.01)²   (61.44 at σ = 0.01;
# the paper's $9.83M corresponds to σ_obs = 4 USDT).  Microprice/half-spread versions
# need order-book depth (file 1-09-1-20.csv); add them here if you have the file.
# ============================================================================
jan = load_days("BTCUSDT", HOLDOUT_JAN2023, verbose=False)
rows13 = []
for d, day in jan.items():
    c = day["close"].to_numpy(); dc = np.diff(c)
    cov1 = np.mean((dc[1:] - dc.mean()) * (dc[:-1] - dc.mean()))
    rows13.append({"date": d, "roll_sigma_usdt": float(np.sqrt(max(0.0, -cov1))),
                   "sd_1s_change_usdt": float(dc.std()), "price": float(c.mean())})
df13 = pd.DataFrame(rows13)
grid = {"roll_median": df13["roll_sigma_usdt"].median(), "sd_1s_change_median": df13["sd_1s_change_usdt"].median(),
        "paper_assumption": 4.0}
cap = {k: {f"delta={dl}": 61.44 * (s / 0.01) ** 2 * (0.05 / dl) ** 4 for dl in (0.02, 0.05, 0.10)}
       for k, s in grid.items()}
print(df13.round(4).to_string(index=False)); print(pd.DataFrame(cap).round(0).to_string())
record("P1.3-T1_sigma_obs", None, "real-data", {"by_day": df13.to_dict("list"), "sigma_choices": grid,
                                                 "capital_usdt": cap},
       note="Replace the $9.83M headline with this grid; label 4 USDT as an upper scenario.")


In [ ]:
# ============================================================================
# CELL P2.5a — PRE-REGISTRATION (run BEFORE P2.5b; commit the file to GitHub)
# The test period (2024) is evaluated only by P2.5b, only if this file exists,
# and every evaluation is counted in prereg/test_set_unlocks.json.
# ============================================================================
PREREG_SPEC = {
    "question": "Does the pre-window transient amplification A_xx add out-of-sample predictive power "
                "for the next 10 minutes' maximum adverse excursion beyond volatility and flow controls?",
    "data": "Binance BTCUSDT 1s klines, panel days (10th and 24th of each month) 2021-2024",
    "unit": "non-overlapping 30-minute blocks; features from [t-10min, t), outcome from (t, t+10min]",
    "outcome": "MAE = max_{s in (t, t+600]} |p_s - p_t| in bp; tail = MAE above train 90th percentile",
    "controls": ["rv_10m", "rv_60m", "abs_ret_10m", "flow_imbalance", "log_volume", "rho", "hour_sin", "hour_cos"],
    "candidate": "log_A_xx",
    "train": "2021-2023", "test": "2024",
    "model": "OLS for MAE; score = fitted MAE for tail AUC",
    "inference": "Clark-West (nested) with Newey-West(4); day-block bootstrap (2,000) for ΔR² and ΔAUC",
    "success": "Clark-West one-sided p < 0.05 AND bootstrap 95% CI of ΔR² excludes 0",
}
spec_txt = json.dumps(PREREG_SPEC, indent=2, sort_keys=True)
PREREG_HASH = hashlib.sha256(spec_txt.encode()).hexdigest()
pr_path = os.path.join(PREREG, "P2_predictive.json")
if not os.path.exists(pr_path):
    with open(pr_path, "w") as f:
        f.write(spec_txt)
    print("pre-registration written:", pr_path)
else:
    old = hashlib.sha256(open(pr_path).read().encode()).hexdigest()
    if old != PREREG_HASH:
        print("!! WARNING: the spec differs from the registered file. Changing it after seeing results "
              "must be disclosed in the paper. Registered hash:", old)
print("spec SHA-256:", PREREG_HASH)


In [ ]:
# ============================================================================
# CELL P2.5b — OUT-OF-SAMPLE PREDICTIVE TEST   (gate G2)   + leakage tests (R2-T8, R2-T10)
# ============================================================================
BLOCK, PRE, POST = 1800, 600, 600

def features_at(p, f, v, t):
    """Uses ONLY indices < t."""
    seg = slice(t - PRE, t); seg60 = slice(max(0, t - 3600), t)
    r = np.diff(p[seg]); r60 = np.diff(p[seg60])
    x = ema_dev(p[max(0, t - PRE - 5 * SPAN):t], SPAN)[-PRE:]
    fn = np.roll(f[seg], -1)
    J = var1_J(x[:-1], fn[:-1])
    A, rho = amp_stats(J)
    hour = ((t % 86400) / 3600.0)
    fs = f[seg]
    return {"rv_10m": r.std(), "rv_60m": r60.std(), "abs_ret_10m": abs(p[t - 1] - p[t - PRE]),
            "flow_imbalance": fs.sum() / (np.abs(fs).sum() + 1e-12), "log_volume": np.log1p(v[seg].sum()),
            "rho": rho, "log_A_xx": np.log(A), "hour_sin": np.sin(2 * np.pi * hour / 24),
            "hour_cos": np.cos(2 * np.pi * hour / 24)}

def outcome_at(p, t):
    return float(np.max(np.abs(p[t + 1:t + 1 + POST] - p[t])))

print("Loading all panel days 2021-2024 ...")
panel = load_days("BTCUSDT", PANEL_DAYS)
rows = []
for d, day in panel.items():
    p, f, v = bars(day, 1)
    for t in range(3600, 86400 - POST - 1, BLOCK):
        r_ = features_at(p, f, v, t); r_["MAE"] = outcome_at(p, t); r_["date"] = d; r_["t"] = t
        rows.append(r_)
dfp = pd.DataFrame(rows)
dfp["year"] = dfp["date"].str[:4].astype(int)
print(f"{len(dfp)} blocks from {dfp['date'].nunique()} days")
# Drop blocks where a feature or the outcome is not finite (e.g. a 10-minute window with no
# trades makes the VAR degenerate). The count is recorded so the paper can report it.
_cols = PREREG_SPEC["controls"] + [PREREG_SPEC["candidate"], "MAE"]
_bad = ~np.isfinite(dfp[_cols].to_numpy(float)).all(1)
DROPPED_BLOCKS = {"n_dropped": int(_bad.sum()), "by_column": {c: int((~np.isfinite(dfp[c].to_numpy(float))).sum()) for c in _cols}}
print("dropped non-finite blocks:", DROPPED_BLOCKS)
dfp = dfp[~_bad].reset_index(drop=True)

# ---- Leakage / look-ahead tests (R2-T8, R2-T10) ---------------------------
Rlk = rng(80); leak_fail = 0
for _ in range(50):
    d = Rlk.choice(list(panel)); p, f, v = bars(panel[d], 1)
    t = int(Rlk.integers(3600, 86400 - POST - 1))
    a = features_at(p, f, v, t)
    p2, f2, v2 = p.copy(), f.copy(), v.copy()
    p2[t:] += Rlk.normal(0, 50, len(p) - t); f2[t:] *= -3; v2[t:] *= 7      # scramble the future
    b = features_at(p2, f2, v2, t)
    leak_fail += any(abs(a[k] - b[k]) > 1e-12 for k in a)
record("R2-T8_no_lookahead", leak_fail == 0, "audit", {"trials": 50, "failures": leak_fail},
       criterion="features unchanged when all data from t onward are scrambled")

# ---- Model fit on train, evaluate on test ---------------------------------
CTRL = PREREG_SPEC["controls"]; CAND = PREREG_SPEC["candidate"]
tr, te = dfp[dfp["year"].isin(TRAIN_YEARS)], dfp[dfp["year"].isin(TEST_YEARS)]

def fit_pred(cols, train, test, y="MAE"):
    X = np.column_stack([np.ones(len(train)), train[cols].to_numpy()])
    b = np.linalg.lstsq(X, train[y].to_numpy(), rcond=None)[0]
    return np.column_stack([np.ones(len(test)), test[cols].to_numpy()]) @ b

def auc(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    ranks = stats.rankdata(np.concatenate([pos, neg]))
    return (ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def clark_west(y, f_small, f_big, lags=4):
    adj = (y - f_small) ** 2 - ((y - f_big) ** 2 - (f_small - f_big) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    tstat = m / np.sqrt(lrv / n)
    return float(tstat), float(1 - stats.norm.cdf(tstat))

def evaluate(df_tr, df_te, cand_col):
    y = df_te["MAE"].to_numpy()
    fb = fit_pred(CTRL, df_tr, df_te); ff = fit_pred(CTRL + [cand_col], df_tr, df_te)
    sst = np.sum((y - df_tr["MAE"].mean()) ** 2)       # OOS R² vs train-mean benchmark
    r2b, r2f = 1 - np.sum((y - fb) ** 2) / sst, 1 - np.sum((y - ff) ** 2) / sst
    tail = (y > np.quantile(df_tr["MAE"], 0.90)).astype(int)
    return {"y": y, "fb": fb, "ff": ff, "tail": tail, "r2_base": r2b, "r2_full": r2f,
            "auc_base": auc(tail, fb), "auc_full": auc(tail, ff)}

def unlock_count():
    pth = os.path.join(PREREG, "test_set_unlocks.json")
    n = json.load(open(pth))["n"] + 1 if os.path.exists(pth) else 1
    json.dump({"n": n, "last_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": PREREG_HASH},
              open(pth, "w"))
    return n

assert os.path.exists(pr_path), "run P2.5a first"
n_unlock = unlock_count()
ev = evaluate(tr, te, CAND)
te_sorted = te.reset_index(drop=True).sort_values(["date", "t"])
order = te_sorted.index.to_numpy()
cw_t, cw_p = clark_west(ev["y"][order], ev["fb"][order], ev["ff"][order])

Rbs = rng(25); days_te = te["date"].to_numpy(); ud = np.unique(days_te)
dR2, dAUC = [], []
for _ in range(2000):
    pick = Rbs.choice(ud, len(ud))
    idx = np.concatenate([np.where(days_te == d)[0] for d in pick])
    y, fb, ff, tl = ev["y"][idx], ev["fb"][idx], ev["ff"][idx], ev["tail"][idx]
    sst = np.sum((y - tr["MAE"].mean()) ** 2)
    dR2.append((np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst)
    dAUC.append(auc(tl, ff) - auc(tl, fb))
ci_r2 = [float(np.nanquantile(dR2, 0.025)), float(np.nanquantile(dR2, 0.975))]
ci_auc = [float(np.nanquantile(dAUC, 0.025)), float(np.nanquantile(dAUC, 0.975))]

# Null feature: AR(1) noise with the candidate's autocorrelation
Rnull = rng(26); c = dfp[CAND].to_numpy(); phi = np.corrcoef(c[1:], c[:-1])[0, 1]
z = np.zeros(len(c)); e = Rnull.normal(size=len(c))
for i in range(1, len(c)):
    z[i] = phi * z[i - 1] + e[i]
dfp["null_feature"] = z
ev_null = evaluate(dfp[dfp["year"].isin(TRAIN_YEARS)], dfp[dfp["year"].isin(TEST_YEARS)], "null_feature")

res25 = {"n_train": len(tr), "n_test": len(te), "r2_base": ev["r2_base"], "r2_full": ev["r2_full"],
         "delta_r2": ev["r2_full"] - ev["r2_base"], "delta_r2_ci": ci_r2,
         "auc_base": ev["auc_base"], "auc_full": ev["auc_full"], "delta_auc_ci": ci_auc,
         "clark_west_t": cw_t, "clark_west_p": cw_p,
         "null_feature_delta_r2": ev_null["r2_full"] - ev_null["r2_base"],
         "prereg_sha256": PREREG_HASH, "test_set_unlocks": n_unlock, "dropped_blocks": DROPPED_BLOCKS}
print(json.dumps(_clean(res25), indent=1))
g2 = cw_p < 0.05 and ci_r2[0] > 0
record("P2.5_predictive_G2", bool(g2), "real-data", res25, criterion=PREREG_SPEC["success"],
       note="Report this result in the paper whatever it is. test_set_unlocks > 1 must be disclosed.")
record("P2.5-T2_null_feature", bool(abs(res25["null_feature_delta_r2"]) < max(abs(ci_r2[1] - ci_r2[0]), 1e-3)),
       "real-data", {"null_feature_delta_r2": res25["null_feature_delta_r2"]},
       criterion="a random feature adds ~0 out-of-sample R² (pipeline sanity)")

# Leave-one-year-out within train (robustness, not the headline)
loyo = {}
for yv in TRAIN_YEARS:
    a_ = dfp[dfp["year"].isin([y for y in TRAIN_YEARS if y != yv])]; b_ = dfp[dfp["year"] == yv]
    e_ = evaluate(a_, b_, CAND); loyo[yv] = e_["r2_full"] - e_["r2_base"]
record("P2.5-T3_leave_one_year_out", None, "real-data", loyo, note="ΔR² by held-out training year.")


In [ ]:
# ============================================================================
# CELL P2.7 — VOL-ONLY ABLATION AND FALSE ALARMS ON REAL OUTCOMES
# Flags set on TRAIN so each flags 10% of blocks; compare on TEST for tail events.
# ============================================================================
thr_w = np.quantile(tr[CAND], 0.90); thr_v = np.quantile(tr["rv_10m"], 0.90)
tail_te = ev["tail"]; wf = (te[CAND].to_numpy() > thr_w); vf = (te["rv_10m"].to_numpy() > thr_v)
def pr(flag, y):
    tp = np.sum(flag & (y == 1)); return {"flag_rate": flag.mean(), "precision": tp / max(flag.sum(), 1),
                                          "recall": tp / max((y == 1).sum(), 1),
                                          "false_alarm_share": 1 - tp / max(flag.sum(), 1)}
res27 = {"wedge_flag": pr(wf, tail_te), "vol_flag": pr(vf, tail_te), "either": pr(wf | vf, tail_te)}
diffs = []
for _ in range(2000):
    pick = Rbs.choice(ud, len(ud)); idx = np.concatenate([np.where(days_te == d)[0] for d in pick])
    y = tail_te[idx]; diffs.append(pr(wf[idx], y)["recall"] - pr(vf[idx], y)["recall"])
res27["recall_diff_wedge_minus_vol_ci"] = [float(np.quantile(diffs, 0.025)), float(np.quantile(diffs, 0.975))]
print(json.dumps(_clean(res27), indent=1))
record("P2.7-T2_vol_ablation", None, "real-data", res27,
       note="If the CI of the recall difference contains 0, write that the wedge flag is not distinguishable "
            "from a volatility flag on real outcomes.")


## R. Code-correctness tests (appendix material)

In [ ]:
# ============================================================================
# CELL R2 — CORE CODE-CORRECTNESS TESTS (appendix material, not evidence)
# R2-T1 ledger identity, R2-T3 Toeplitz positive definiteness, R2-T5 merged-measure
# invariance, R2-T9 data integrity log.
# ============================================================================
# R2-T1: execution at P* + Σ_{s<t} h_{t-s} v_s + ½ h_0 v_t + c·sign(v_t).
# For a flat round trip (Σ v = 0) the cash loss equals ½ vᵀ T_h v + c ||v||₁ with T_h[t,s] = h_|t-s|.
Rr = rng(90); worst = 0.0
for _ in range(1000):
    K = Rr.integers(1, 4); r = Rr.uniform(0.1, 0.9, K); w = Rr.uniform(0, 0.15, K)
    if not is_stable(w, r, A_DEF, KAPPA_DEF):
        continue
    n = int(Rr.integers(4, 40)); h = impulse(w, r, A_DEF, KAPPA_DEF, n); c = Rr.uniform(0, 0.1)
    v = Rr.normal(size=n); v -= v.mean()
    cash = 0.0
    for t in range(n):
        px = 100.0 + sum(h[t - s] * v[s] for s in range(t)) + 0.5 * h[0] * v[t] + c * np.sign(v[t])
        cash -= v[t] * px                       # buy v_t > 0 pays, sell receives
    loss = -cash                                # Σ v = 0, so terminal inventory is flat
    T = linalg.toeplitz(h)
    worst = max(worst, abs(loss - (0.5 * v @ T @ v + c * np.abs(v).sum())))
record("R2-T1_ledger_identity", worst < 1e-9, "audit", {"max_abs_error": worst},
       criterion="|ledger loss - (½vᵀT v + c||v||₁)| < 1e-9 on 1,000 random probes")

# R2-T3: Toeplitz of a positive-economy response is positive definite
min_eig = np.inf
for _ in range(1000):
    K = Rr.integers(1, 4); r = Rr.uniform(0.1, 0.9, K); w = Rr.uniform(0, 0.15, K)
    if not is_stable(w, r, A_DEF, KAPPA_DEF):
        continue
    h = impulse(w, r, A_DEF, KAPPA_DEF, 60)
    min_eig = min(min_eig, np.linalg.eigvalsh(linalg.toeplitz(h)).min())
record("R2-T3_toeplitz_pd", min_eig > 0, "audit", {"min_eigenvalue": min_eig},
       criterion="smallest eigenvalue of T_h > 0 over 1,000 stable positive economies")

# R2-T5: same merged measure -> identical responses; different measure -> different
h1 = impulse([0.06], [0.7], A_DEF, KAPPA_DEF, 200)
h2 = impulse([0.02, 0.04], [0.7, 0.7], A_DEF, KAPPA_DEF, 200)            # split, same rate
h3 = impulse([0.03, 0.03], [0.65, 0.75], A_DEF, KAPPA_DEF, 200)          # different measure
record("R2-T5_merged_invariance", bool(np.max(np.abs(h1 - h2)) < 1e-12 and np.max(np.abs(h1 - h3)) > 1e-6), "audit",
       {"same_measure_max_diff": np.max(np.abs(h1 - h2)), "different_measure_max_diff": np.max(np.abs(h1 - h3))},
       criterion="identical for equal merged measures, different otherwise")

# R2-T9: every downloaded file was checked against Binance's published SHA-256
dl = pd.DataFrame(DOWNLOAD_LOG)
if len(dl):
    dl.to_csv(os.path.join(RESULTS, "data_manifest_sha256.csv"), index=False)
record("R2-T9_data_integrity", True if len(dl) or glob.glob(os.path.join(DATA, "*.parquet")) else None, "audit",
       {"files_downloaded_this_session": len(dl), "cached_files": len(glob.glob(os.path.join(DATA, "*.parquet")))},
       criterion="every downloaded archive matched its official .CHECKSUM (load_day raises otherwise)",
       note="results/data_manifest_sha256.csv lists the hashes; put it in the replication package.")


## Z. Number trace, summary and download
A4 lists every number in the papers that no result file supports. Rewrite or delete each one.

In [ ]:
# ============================================================================
# CELL A4 — DO THE PAPERS' NUMBERS COME FROM THESE RESULTS?  (P2.1-T1)
# Lists every number with >= 3 significant figures in each paper's abstract and
# main text that does not match any value in results/*.json.  Each orphan must
# be traced to a results file or removed.
# ============================================================================
def _all_values(o, out):
    if isinstance(o, dict):
        for v in o.values():
            _all_values(v, out)
    elif isinstance(o, list):
        for v in o:
            _all_values(v, out)
    elif isinstance(o, (int, float)) and o is not None and not isinstance(o, bool):
        out.append(float(o))
    return out

vals = []
for fp in glob.glob(os.path.join(RESULTS, "*.json")):
    vals += _all_values(json.load(open(fp)), [])
vals = np.array(vals) if vals else np.array([np.nan])

def _matches(xstr):
    x = float(xstr.replace(",", ""))
    dec = len(xstr.split(".")[1]) if "." in xstr else 0
    tol = 0.5 * 10 ** (-dec)
    cands = [vals, vals * 100, vals / 100]                     # allow percent conversions
    return any(np.any(np.abs(c - x) <= tol + 1e-12) for c in cands)

orph = {}
for label, path in [("paper1", PAPER1_PDF), ("paper2", PAPER2_PDF)]:
    if not path:
        continue
    pages = pdf_pages_text(path)
    main = []
    for p in pages:
        head = "\n".join(p.strip().split("\n")[:3])
        if re.search(r"^\s*(Internet Appendix|References|Appendix)\s*$|^\s*(Internet Appendix|References)\b", head, re.M) and len(main) > 5:
            break
        main.append(p)
    text = " ".join(main)
    nums = set(re.findall(r"(?<![\w.])(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+\.\d{2,})(?![\w])", text))
    nums = {n for n in nums if len(re.sub(r"[^0-9]", "", n).lstrip("0")) >= 3}
    orph[label] = sorted(n for n in nums if not _matches(n))
    print(f"{label}: {len(nums)} numbers in main text, {len(orph[label])} not produced by any result file")
    print("   e.g.", orph[label][:40])
record("P2.1-T1_number_trace", all(len(v) == 0 for v in orph.values()), "audit",
       {k: {"n_orphans": len(v), "orphans": v} for k, v in orph.items()},
       criterion="every 3+ significant-figure number in abstract/main text appears in results/*.json",
       note="Expected to FAIL now. Rewrite each number from these result files; delete what no file supports.")


In [ ]:
# ============================================================================
# CELL Z — SUMMARY TABLE AND DOWNLOAD
# ============================================================================
rows = []
for fp in sorted(glob.glob(os.path.join(RESULTS, "*.json"))):
    r = json.load(open(fp))
    rows.append({"test": r["test_id"], "status": r["status"], "kind": r["kind"], "criterion": r["criterion"][:90]})
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RESULTS, "SUMMARY.csv"), index=False)
pd.set_option("display.max_colwidth", 95); pd.set_option("display.width", 220)
print(summary.to_string(index=False))
print("\nCounts:", summary.groupby(["kind", "status"]).size().to_dict())

zip_path = os.path.join(BASE, "ceiling_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "*")) + glob.glob(os.path.join(PREREG, "*")):
        z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)


## After you run it: what each result decides
| Test | Gate | If PASS | If FAIL |
|---|---|---|---|
| A1 hard-coded scan | before posting anything | — | delete every typed-in number from both papers; replace with values from `results/` |
| S1-T1, S1-T2, S2-T1, S3-T1 | G0 | papers are split and clean | finish Phase 0 (split, recompile from LaTeX, anonymize) |
| P2.2-T2 artifact null | **G1** | keep "empirical amplification" as a headline | say amplification is not distinguishable from pipeline artifacts; lead with theory (T6) |
| P2.2-T4 selection | G1 | — | if only outcome-selected windows amplify, drop the crash-window claim |
| T6-E overshoot | supports T6 | cite flow-held-fixed overshoot as evidence for the depth channel | present T6 as theory; say klines cannot separate the channel (needs depth data) |
| P2.5 predictive | **G2** | abstract may say "predicts" with the CI | abstract must not say "predicts"; report the null result in the body |
| P2.7 vol ablation | G2 | wedge adds information beyond volatility | write that it is not distinguishable from a volatility flag |
| P1.6 EMA placebo | P1 | — | write "no evidence of feedback beyond EMA mechanics" |
| P1.5 coverage table | P1 | — | call every estimator outside 93–97% coverage biased |
| T1–T9 | proofs | write the proof from the route in the plan doc | the statement is false as written; change it |
| P2.1-T1 number trace | G4 | every number traced | rewrite or delete each orphan number |

Send me `ceiling_results.zip` (downloaded by the last cell) and I will check the results and draft the paper text from them.